# Парсинг данных новостных сообществ социальной сети ВКонтакте

## Парсинг публикаций

### Написание функции по извлечению данных публикаций с помощью VK API

In [83]:
pip install vk_api

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
# api_token = #'Вставьте свой ключ доступа'

# используемый личный ключ скрыт из ячейки в целях безопасности аккаунта в соцсети

In [ ]:
# импорт библиотек
import vk_api
from datetime import datetime
import time
import pandas as pd

# инициализация функции для извлечения публикаций
def get_posts(api_token, group_id, count=100, start_offset=5000):
  '''
  Получает тексты постов и пользовательские реакции из сообществ с сайта https://vk.com.
  Собирает посты, пока не дойдет до даты 01.01.2020, либо пока не закончатся посты.

  Параметры:
    api_token: ключ доступа для выполнения запросов к VK API
    group_id: уникальный идентификатор сообщества, начинающийся с '-'
    count: число выгружаемых постов за 1 итерацию
    start_offset: начальная позиция со смещением (сколько последних постов пропускаем перед сбором данных)
  Вывод функции:
    Возвращает all_posts – список из словарей; в каждом словаре содержится информация
                           о конкретной публикации.
  '''
  # инициализация VK API
  vk_session = vk_api.VkApi(token=api_token)
  vk = vk_session.get_api()

  # инициализация словаря, смещения, целевой даты и флага ее достижения
  all_posts = []
  offset = start_offset
  target_date = datetime(2020, 1, 1).timestamp()
  target_flag = False

  print(f'Начинаем сбор с offset = {offset}')
  print(f'Цель: дойти до 01.01.2020')
  print('-' * 50)

  # пока не достигнута целевая дата, цикл в работе
  while not target_flag:
    # отправка запроса к новостной 'стене' сообщества
    response = vk.wall.get(owner_id=group_id,
                           filter='owner',
                           count=count,
                           offset=offset)
    # пустой ответ на запрос сигнализирует об отсутствии постов
    if not response.get('items'):
      print('Постов больше нет')
      break
    # извлечение даты поста из ответа на запрос
    for post in response['items']:
      post_date = post['date']
      post_date_string = datetime.fromtimestamp(post_date).strftime('%Y-%m-%d')
      post_year = datetime.fromtimestamp(post_date).year
      # проверка на достижении целевой даты 01.01.2020
      if post_date <= target_date:
        target_flag = True
        if post_year == 2020 or post_year < 2020:
          print(f'\n- Достигли {post_year} года -')
          print(f'Последний пост: {post['id']} от {post_date_string}')
          break
      # извлечение данных из ответа на запрос в виде словаря и присоединение их к списку со всеми постами
      else:
        all_posts.append({'post_id': post['id'],
                          'date': post_date_string,
                          'text': post['text'],
                          'likes': post.get('likes', {}).get('count', 0),
                          'comments_count': post.get('comments', {}).get('count', 0),
                          'reposts': post.get('reposts', {}).get('count', 0),
                          'views': post.get('views', {}).get('count', 0) if 'views' in post else 0})
    # завершение цикла в случае достижения целевой даты 01.01.2020
    if target_flag:
      break

    print(f'Обработано offset {offset}, собрано {len(all_posts)} постов (последний: {post_date_string})')
    # если цикл продолжается, значение смещения обновляется на число собранных постов (100)
    offset += 100
    # задержка алгоритма для избежания превышения установленных лимитов запросов (rps)
    time.sleep(0.1)

  return all_posts

### Применение функции для выгрузки данных из сообществ

Сообщества университетов Москвы, имеющие суммарный индекс > 1. Сайт: https://brandanalytics.ru/university-rankings/integral/#scroll_anchor

In [ ]:
# ВШЭ
group_id = '-25205856'
name = 'hse.csv'

posts = get_posts(api_token, group_id, start_offset=300)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 300
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 300, собрано 100 постов (последний: 2026-01-13)
Обработано offset 400, собрано 200 постов (последний: 2025-12-08)
Обработано offset 500, собрано 300 постов (последний: 2025-11-13)
Обработано offset 600, собрано 400 постов (последний: 2025-10-18)
Обработано offset 700, собрано 500 постов (последний: 2025-09-26)
Обработано offset 800, собрано 600 постов (последний: 2025-09-01)
Обработано offset 900, собрано 700 постов (последний: 2025-07-24)
Обработано offset 1000, собрано 800 постов (последний: 2025-06-27)
Обработано offset 1100, собрано 900 постов (последний: 2025-05-30)
Обработано offset 1200, собрано 1000 постов (последний: 2025-05-07)
Обработано offset 1300, собрано 1100 постов (последний: 2025-04-15)
Обработано offset 1400, собрано 1200 постов (последний: 2025-03-18)
Обработано offset 1500, собрано 1300 постов (последний: 2025-02-20)
Обработано offset 1600, со

In [ ]:
# МАИ
group_id = '-50409684'
name = 'mai.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-22)
Обработано offset 100, собрано 200 постов (последний: 2026-03-20)
Обработано offset 200, собрано 300 постов (последний: 2026-01-27)
Обработано offset 300, собрано 400 постов (последний: 2025-12-19)
Обработано offset 400, собрано 500 постов (последний: 2025-11-17)
Обработано offset 500, собрано 600 постов (последний: 2025-10-14)
Обработано offset 600, собрано 700 постов (последний: 2025-09-10)
Обработано offset 700, собрано 800 постов (последний: 2025-08-11)
Обработано offset 800, собрано 900 постов (последний: 2025-07-09)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-06)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-09)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-10)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-07)
Обработано offset 1300, собрано 1

In [ ]:
# МГУ
group_id = '-78019879'
name = 'mgu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-10)
Обработано offset 100, собрано 200 постов (последний: 2026-03-20)
Обработано offset 200, собрано 300 постов (последний: 2026-02-27)
Обработано offset 300, собрано 400 постов (последний: 2026-02-07)
Обработано offset 400, собрано 500 постов (последний: 2026-01-19)
Обработано offset 500, собрано 600 постов (последний: 2025-12-19)
Обработано offset 600, собрано 700 постов (последний: 2025-11-28)
Обработано offset 700, собрано 800 постов (последний: 2025-11-12)
Обработано offset 800, собрано 900 постов (последний: 2025-10-16)
Обработано offset 900, собрано 1000 постов (последний: 2025-09-22)
Обработано offset 1000, собрано 1100 постов (последний: 2025-08-29)
Обработано offset 1100, собрано 1200 постов (последний: 2025-08-05)
Обработано offset 1200, собрано 1300 постов (последний: 2025-07-16)
Обработано offset 1300, собрано 1

In [ ]:
# РЭУ ПЛЕХАНОВА
group_id = '-1177'
name = 'reu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-29)
Обработано offset 100, собрано 200 постов (последний: 2026-02-18)
Обработано offset 200, собрано 300 постов (последний: 2025-12-15)
Обработано offset 300, собрано 400 постов (последний: 2025-09-14)
Обработано offset 400, собрано 500 постов (последний: 2025-07-17)
Обработано offset 500, собрано 600 постов (последний: 2025-05-21)
Обработано offset 600, собрано 700 постов (последний: 2025-04-09)
Обработано offset 700, собрано 800 постов (последний: 2025-03-06)
Обработано offset 800, собрано 900 постов (последний: 2025-01-21)
Обработано offset 900, собрано 1000 постов (последний: 2024-12-12)
Обработано offset 1000, собрано 1100 постов (последний: 2024-11-04)
Обработано offset 1100, собрано 1200 постов (последний: 2024-09-24)
Обработано offset 1200, собрано 1300 постов (последний: 2024-08-19)
Обработано offset 1300, собрано 1

In [ ]:
# МФТИ
group_id = '-932'
name = 'mfti.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-17)
Обработано offset 100, собрано 200 постов (последний: 2026-02-02)
Обработано offset 200, собрано 300 постов (последний: 2025-12-10)
Обработано offset 300, собрано 400 постов (последний: 2025-11-14)
Обработано offset 400, собрано 500 постов (последний: 2025-10-16)
Обработано offset 500, собрано 600 постов (последний: 2025-09-18)
Обработано offset 600, собрано 700 постов (последний: 2025-08-22)
Обработано offset 700, собрано 800 постов (последний: 2025-07-16)
Обработано offset 800, собрано 900 постов (последний: 2025-06-09)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-16)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-17)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-25)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-03)
Обработано offset 1300, собрано 1

In [ ]:
# МИРЭА
group_id = '-1388'
name = 'mirea.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-20)
Обработано offset 100, собрано 200 постов (последний: 2026-04-05)
Обработано offset 200, собрано 300 постов (последний: 2026-03-16)
Обработано offset 300, собрано 400 постов (последний: 2026-02-26)
Обработано offset 400, собрано 500 постов (последний: 2026-02-09)
Обработано offset 500, собрано 600 постов (последний: 2026-01-20)
Обработано offset 600, собрано 700 постов (последний: 2025-12-27)
Обработано offset 700, собрано 800 постов (последний: 2025-12-06)
Обработано offset 800, собрано 900 постов (последний: 2025-11-17)
Обработано offset 900, собрано 1000 постов (последний: 2025-10-30)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-13)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-25)
Обработано offset 1200, собрано 1300 постов (последний: 2025-09-05)
Обработано offset 1300, собрано 1

In [ ]:
# РАНХ
group_id = '-5398'
name = 'ranepa.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-19)
Обработано offset 100, собрано 200 постов (последний: 2026-03-31)
Обработано offset 200, собрано 300 постов (последний: 2026-03-13)
Обработано offset 300, собрано 400 постов (последний: 2026-02-20)
Обработано offset 400, собрано 500 постов (последний: 2026-02-09)
Обработано offset 500, собрано 600 постов (последний: 2026-01-22)
Обработано offset 600, собрано 700 постов (последний: 2025-12-25)
Обработано offset 700, собрано 800 постов (последний: 2025-12-08)
Обработано offset 800, собрано 900 постов (последний: 2025-11-20)
Обработано offset 900, собрано 1000 постов (последний: 2025-11-01)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-14)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-26)
Обработано offset 1200, собрано 1300 постов (последний: 2025-09-13)
Обработано offset 1300, собрано 1

In [ ]:
# МИСИС
group_id = '-62258607'
name = 'misis.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-05)
Обработано offset 100, собрано 200 постов (последний: 2026-03-05)
Обработано offset 200, собрано 300 постов (последний: 2026-02-08)
Обработано offset 300, собрано 400 постов (последний: 2026-01-09)
Обработано offset 400, собрано 500 постов (последний: 2025-12-08)
Обработано offset 500, собрано 600 постов (последний: 2025-11-13)
Обработано offset 600, собрано 700 постов (последний: 2025-10-17)
Обработано offset 700, собрано 800 постов (последний: 2025-09-22)
Обработано offset 800, собрано 900 постов (последний: 2025-08-29)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-30)
Обработано offset 1000, собрано 1100 постов (последний: 2025-06-26)
Обработано offset 1100, собрано 1200 постов (последний: 2025-05-27)
Обработано offset 1200, собрано 1300 постов (последний: 2025-04-25)
Обработано offset 1300, собрано 1

In [ ]:
# бауманка
group_id = '-40427933'
name = 'bmstu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-11)
Обработано offset 100, собрано 200 постов (последний: 2026-03-10)
Обработано offset 200, собрано 300 постов (последний: 2026-02-05)
Обработано offset 300, собрано 400 постов (последний: 2025-12-28)
Обработано offset 400, собрано 500 постов (последний: 2025-11-28)
Обработано offset 500, собрано 600 постов (последний: 2025-10-29)
Обработано offset 600, собрано 700 постов (последний: 2025-09-30)
Обработано offset 700, собрано 800 постов (последний: 2025-08-26)
Обработано offset 800, собрано 900 постов (последний: 2025-07-21)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-20)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-21)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-18)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-21)
Обработано offset 1300, собрано 1

In [ ]:
# рудн
group_id = '-1711'
name = 'rudn.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-06)
Обработано offset 100, собрано 200 постов (последний: 2026-03-03)
Обработано offset 200, собрано 300 постов (последний: 2026-01-25)
Обработано offset 300, собрано 400 постов (последний: 2025-12-04)
Обработано offset 400, собрано 500 постов (последний: 2025-10-28)
Обработано offset 500, собрано 600 постов (последний: 2025-09-17)
Обработано offset 600, собрано 700 постов (последний: 2025-08-16)
Обработано offset 700, собрано 800 постов (последний: 2025-07-17)
Обработано offset 800, собрано 900 постов (последний: 2025-06-20)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-19)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-16)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-17)
Обработано offset 1200, собрано 1300 постов (последний: 2025-02-08)
Обработано offset 1300, собрано 1

In [ ]:
# сеченов
group_id = '-65464437'
name = 'sechenov.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-03)
Обработано offset 100, собрано 200 постов (последний: 2026-03-03)
Обработано offset 200, собрано 300 постов (последний: 2026-01-29)
Обработано offset 300, собрано 400 постов (последний: 2025-12-27)
Обработано offset 400, собрано 500 постов (последний: 2025-11-30)
Обработано offset 500, собрано 600 постов (последний: 2025-10-24)
Обработано offset 600, собрано 700 постов (последний: 2025-09-25)
Обработано offset 700, собрано 800 постов (последний: 2025-08-28)
Обработано offset 800, собрано 900 постов (последний: 2025-07-22)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-19)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-22)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-16)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-11)
Обработано offset 1300, собрано 1

In [ ]:
# мгимо
group_id = '-26555975'
name = 'mgimo.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-06)
Обработано offset 100, собрано 200 постов (последний: 2026-03-10)
Обработано offset 200, собрано 300 постов (последний: 2026-02-12)
Обработано offset 300, собрано 400 постов (последний: 2025-12-19)
Обработано offset 400, собрано 500 постов (последний: 2025-11-24)
Обработано offset 500, собрано 600 постов (последний: 2025-10-28)
Обработано offset 600, собрано 700 постов (последний: 2025-10-07)
Обработано offset 700, собрано 800 постов (последний: 2025-09-19)
Обработано offset 800, собрано 900 постов (последний: 2025-08-26)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-18)
Обработано offset 1000, собрано 1100 постов (последний: 2025-07-02)
Обработано offset 1100, собрано 1200 постов (последний: 2025-06-15)
Обработано offset 1200, собрано 1300 постов (последний: 2025-05-24)
Обработано offset 1300, собрано 1

In [ ]:
# финашка
group_id = '-6319'
name = 'fa.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-08)
Обработано offset 100, собрано 200 постов (последний: 2026-03-13)
Обработано offset 200, собрано 300 постов (последний: 2026-02-12)
Обработано offset 300, собрано 400 постов (последний: 2026-01-02)
Обработано offset 400, собрано 500 постов (последний: 2025-12-08)
Обработано offset 500, собрано 600 постов (последний: 2025-11-13)
Обработано offset 600, собрано 700 постов (последний: 2025-10-16)
Обработано offset 700, собрано 800 постов (последний: 2025-09-18)
Обработано offset 800, собрано 900 постов (последний: 2025-08-25)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-14)
Обработано offset 1000, собрано 1100 постов (последний: 2025-06-04)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-28)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-29)
Обработано offset 1300, собрано 1

In [ ]:
# мифи
group_id = '-69589815'
name = 'mephi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-18)
Обработано offset 100, собрано 200 постов (последний: 2026-04-04)
Обработано offset 200, собрано 300 постов (последний: 2026-03-19)
Обработано offset 300, собрано 400 постов (последний: 2026-02-28)
Обработано offset 400, собрано 500 постов (последний: 2026-02-10)
Обработано offset 500, собрано 600 постов (последний: 2026-01-18)
Обработано offset 600, собрано 700 постов (последний: 2025-12-17)
Обработано offset 700, собрано 800 постов (последний: 2025-11-27)
Обработано offset 800, собрано 900 постов (последний: 2025-11-08)
Обработано offset 900, собрано 1000 постов (последний: 2025-10-15)
Обработано offset 1000, собрано 1100 постов (последний: 2025-09-26)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-08)
Обработано offset 1200, собрано 1300 постов (последний: 2025-08-20)
Обработано offset 1300, собрано 1

In [ ]:
# кутафина
group_id = '-65417'
name = 'kutafina.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-17)
Обработано offset 100, собрано 200 постов (последний: 2026-03-30)
Обработано offset 200, собрано 300 постов (последний: 2026-03-12)
Обработано offset 300, собрано 400 постов (последний: 2026-02-24)
Обработано offset 400, собрано 500 постов (последний: 2026-02-08)
Обработано offset 500, собрано 600 постов (последний: 2026-01-19)
Обработано offset 600, собрано 700 постов (последний: 2025-12-26)
Обработано offset 700, собрано 800 постов (последний: 2025-12-07)
Обработано offset 800, собрано 900 постов (последний: 2025-11-19)
Обработано offset 900, собрано 1000 постов (последний: 2025-10-26)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-01)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-12)
Обработано offset 1200, собрано 1300 постов (последний: 2025-08-27)
Обработано offset 1300, собрано 1

In [ ]:
# Гитис
group_id = '-145886776'
name = 'gitis.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-15)
Обработано offset 100, собрано 200 постов (последний: 2026-03-19)
Обработано offset 200, собрано 300 постов (последний: 2026-02-18)
Обработано offset 300, собрано 400 постов (последний: 2026-01-19)
Обработано offset 400, собрано 500 постов (последний: 2025-12-11)
Обработано offset 500, собрано 600 постов (последний: 2025-11-14)
Обработано offset 600, собрано 700 постов (последний: 2025-10-13)
Обработано offset 700, собрано 800 постов (последний: 2025-09-09)
Обработано offset 800, собрано 900 постов (последний: 2025-07-18)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-16)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-13)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-31)
Обработано offset 1200, собрано 1300 постов (последний: 2025-02-12)
Обработано offset 1300, собрано 1

In [ ]:
# губкин
group_id = '-71938736'
name = 'gubkin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-14)
Обработано offset 100, собрано 200 постов (последний: 2026-03-24)
Обработано offset 200, собрано 300 постов (последний: 2026-03-06)
Обработано offset 300, собрано 400 постов (последний: 2026-02-17)
Обработано offset 400, собрано 500 постов (последний: 2026-01-26)
Обработано offset 500, собрано 600 постов (последний: 2025-12-22)
Обработано offset 600, собрано 700 постов (последний: 2025-11-29)
Обработано offset 700, собрано 800 постов (последний: 2025-11-15)
Обработано offset 800, собрано 900 постов (последний: 2025-10-31)
Обработано offset 900, собрано 1000 постов (последний: 2025-10-09)
Обработано offset 1000, собрано 1100 постов (последний: 2025-09-13)
Обработано offset 1100, собрано 1200 постов (последний: 2025-08-14)
Обработано offset 1200, собрано 1300 постов (последний: 2025-07-09)
Обработано offset 1300, собрано 1

In [ ]:
# рниму
group_id = '-53765914'
name = 'rsmu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-03)
Обработано offset 100, собрано 200 постов (последний: 2026-03-08)
Обработано offset 200, собрано 300 постов (последний: 2026-02-03)
Обработано offset 300, собрано 400 постов (последний: 2025-12-18)
Обработано offset 400, собрано 500 постов (последний: 2025-11-25)
Обработано offset 500, собрано 600 постов (последний: 2025-11-05)
Обработано offset 600, собрано 700 постов (последний: 2025-10-13)
Обработано offset 700, собрано 800 постов (последний: 2025-09-17)
Обработано offset 800, собрано 900 постов (последний: 2025-08-14)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-17)
Обработано offset 1000, собрано 1100 постов (последний: 2025-06-20)
Обработано offset 1100, собрано 1200 постов (последний: 2025-05-26)
Обработано offset 1200, собрано 1300 постов (последний: 2025-05-03)
Обработано offset 1300, собрано 1

In [ ]:
# мэи
group_id = '-51345945'
name = 'mpei.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-14)
Обработано offset 100, собрано 200 постов (последний: 2026-03-18)
Обработано offset 200, собрано 300 постов (последний: 2026-02-17)
Обработано offset 300, собрано 400 постов (последний: 2026-01-12)
Обработано offset 400, собрано 500 постов (последний: 2025-12-01)
Обработано offset 500, собрано 600 постов (последний: 2025-11-03)
Обработано offset 600, собрано 700 постов (последний: 2025-10-09)
Обработано offset 700, собрано 800 постов (последний: 2025-09-16)
Обработано offset 800, собрано 900 постов (последний: 2025-08-25)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-31)
Обработано offset 1000, собрано 1100 постов (последний: 2025-07-08)
Обработано offset 1100, собрано 1200 постов (последний: 2025-06-10)
Обработано offset 1200, собрано 1300 постов (последний: 2025-05-16)
Обработано offset 1300, собрано 1

In [ ]:
# мпгу
group_id = '-30321356'
name = 'mpgu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-22)
Обработано offset 100, собрано 200 постов (последний: 2026-04-09)
Обработано offset 200, собрано 300 постов (последний: 2026-03-28)
Обработано offset 300, собрано 400 постов (последний: 2026-03-17)
Обработано offset 400, собрано 500 постов (последний: 2026-03-05)
Обработано offset 500, собрано 600 постов (последний: 2026-02-20)
Обработано offset 600, собрано 700 постов (последний: 2026-02-06)
Обработано offset 700, собрано 800 постов (последний: 2026-01-25)
Обработано offset 800, собрано 900 постов (последний: 2026-01-08)
Обработано offset 900, собрано 1000 постов (последний: 2025-12-22)
Обработано offset 1000, собрано 1100 постов (последний: 2025-12-11)
Обработано offset 1100, собрано 1200 постов (последний: 2025-11-28)
Обработано offset 1200, собрано 1300 постов (последний: 2025-11-16)
Обработано offset 1300, собрано 1

In [ ]:
# вгик
group_id = '-199859404'
name = 'vgik.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-22)
Обработано offset 100, собрано 200 постов (последний: 2026-04-10)
Обработано offset 200, собрано 300 постов (последний: 2026-03-27)
Обработано offset 300, собрано 400 постов (последний: 2026-03-15)
Обработано offset 400, собрано 500 постов (последний: 2026-02-26)
Обработано offset 500, собрано 600 постов (последний: 2026-02-10)
Обработано offset 600, собрано 700 постов (последний: 2026-01-26)
Обработано offset 700, собрано 800 постов (последний: 2025-12-29)
Обработано offset 800, собрано 900 постов (последний: 2025-12-05)
Обработано offset 900, собрано 1000 постов (последний: 2025-11-10)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-10)
Обработано offset 1100, собрано 1200 постов (последний: 2025-08-26)
Обработано offset 1200, собрано 1300 постов (последний: 2025-07-07)
Обработано offset 1300, собрано 1

In [ ]:
# мгпу
group_id = '-3983'
name = 'mgpu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-17)
Обработано offset 100, собрано 200 постов (последний: 2026-01-14)
Обработано offset 200, собрано 300 постов (последний: 2025-11-12)
Обработано offset 300, собрано 400 постов (последний: 2025-10-02)
Обработано offset 400, собрано 500 постов (последний: 2025-08-22)
Обработано offset 500, собрано 600 постов (последний: 2025-07-16)
Обработано offset 600, собрано 700 постов (последний: 2025-06-04)
Обработано offset 700, собрано 800 постов (последний: 2025-05-05)
Обработано offset 800, собрано 900 постов (последний: 2025-04-01)
Обработано offset 900, собрано 1000 постов (последний: 2025-02-21)
Обработано offset 1000, собрано 1100 постов (последний: 2025-01-16)
Обработано offset 1100, собрано 1200 постов (последний: 2024-12-04)
Обработано offset 1200, собрано 1300 постов (последний: 2024-10-28)
Обработано offset 1300, собрано 1

In [ ]:
# рггу
group_id = '-16479782'
name = 'rggu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-04)
Обработано offset 100, собрано 200 постов (последний: 2026-03-03)
Обработано offset 200, собрано 300 постов (последний: 2025-12-12)
Обработано offset 300, собрано 400 постов (последний: 2025-11-20)
Обработано offset 400, собрано 500 постов (последний: 2025-10-21)
Обработано offset 500, собрано 600 постов (последний: 2025-09-22)
Обработано offset 600, собрано 700 постов (последний: 2025-08-21)
Обработано offset 700, собрано 800 постов (последний: 2025-07-20)
Обработано offset 800, собрано 900 постов (последний: 2025-06-26)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-27)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-05)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-20)
Обработано offset 1200, собрано 1300 постов (последний: 2025-04-05)
Обработано offset 1300, собрано 1

In [ ]:
# миэт
group_id = '-33509'
name = 'miet.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-09)
Обработано offset 100, собрано 200 постов (последний: 2026-03-17)
Обработано offset 200, собрано 300 постов (последний: 2026-02-13)
Обработано offset 300, собрано 400 постов (последний: 2026-01-15)
Обработано offset 400, собрано 500 постов (последний: 2025-12-13)
Обработано offset 500, собрано 600 постов (последний: 2025-11-27)
Обработано offset 600, собрано 700 постов (последний: 2025-11-04)
Обработано offset 700, собрано 800 постов (последний: 2025-10-14)
Обработано offset 800, собрано 900 постов (последний: 2025-09-22)
Обработано offset 900, собрано 1000 постов (последний: 2025-08-31)
Обработано offset 1000, собрано 1100 постов (последний: 2025-08-05)
Обработано offset 1100, собрано 1200 постов (последний: 2025-07-12)
Обработано offset 1200, собрано 1300 постов (последний: 2025-06-21)
Обработано offset 1300, собрано 1

In [ ]:
# МСХА имени К.А. Тимирязева, академика РАН
group_id = '-146300938'
name = 'timacad.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-10)
Обработано offset 100, собрано 200 постов (последний: 2026-03-14)
Обработано offset 200, собрано 300 постов (последний: 2026-02-14)
Обработано offset 300, собрано 400 постов (последний: 2026-01-18)
Обработано offset 400, собрано 500 постов (последний: 2025-12-13)
Обработано offset 500, собрано 600 постов (последний: 2025-11-18)
Обработано offset 600, собрано 700 постов (последний: 2025-10-24)
Обработано offset 700, собрано 800 постов (последний: 2025-09-27)
Обработано offset 800, собрано 900 постов (последний: 2025-09-02)
Обработано offset 900, собрано 1000 постов (последний: 2025-08-01)
Обработано offset 1000, собрано 1100 постов (последний: 2025-06-26)
Обработано offset 1100, собрано 1200 постов (последний: 2025-05-24)
Обработано offset 1200, собрано 1300 постов (последний: 2025-04-18)
Обработано offset 1300, собрано 1

In [ ]:
# гуу
group_id = '-23628595'
name = 'guu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-17)
Обработано offset 100, собрано 200 постов (последний: 2026-04-01)
Обработано offset 200, собрано 300 постов (последний: 2026-03-16)
Обработано offset 300, собрано 400 постов (последний: 2026-02-21)
Обработано offset 400, собрано 500 постов (последний: 2026-01-30)
Обработано offset 500, собрано 600 постов (последний: 2026-01-02)
Обработано offset 600, собрано 700 постов (последний: 2025-12-03)
Обработано offset 700, собрано 800 постов (последний: 2025-11-11)
Обработано offset 800, собрано 900 постов (последний: 2025-10-13)
Обработано offset 900, собрано 1000 постов (последний: 2025-08-22)
Обработано offset 1000, собрано 1100 постов (последний: 2025-07-02)
Обработано offset 1100, собрано 1200 постов (последний: 2025-05-14)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-22)
Обработано offset 1300, собрано 1

In [ ]:
# мгсу
group_id = '-97639426'
name = 'mgsu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-17)
Обработано offset 100, собрано 200 постов (последний: 2026-04-02)
Обработано offset 200, собрано 300 постов (последний: 2026-03-18)
Обработано offset 300, собрано 400 постов (последний: 2026-02-28)
Обработано offset 400, собрано 500 постов (последний: 2026-02-09)
Обработано offset 500, собрано 600 постов (последний: 2026-01-21)
Обработано offset 600, собрано 700 постов (последний: 2025-12-25)
Обработано offset 700, собрано 800 постов (последний: 2025-12-05)
Обработано offset 800, собрано 900 постов (последний: 2025-11-21)
Обработано offset 900, собрано 1000 постов (последний: 2025-11-03)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-17)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-30)
Обработано offset 1200, собрано 1300 постов (последний: 2025-09-17)
Обработано offset 1300, собрано 1

In [ ]:
# косыгина
group_id = '-38924'
name = 'rguk.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-12)
Обработано offset 100, собрано 200 постов (последний: 2026-03-16)
Обработано offset 200, собрано 300 постов (последний: 2026-02-20)
Обработано offset 300, собрано 400 постов (последний: 2026-01-28)
Обработано offset 400, собрано 500 постов (последний: 2025-12-22)
Обработано offset 500, собрано 600 постов (последний: 2025-11-27)
Обработано offset 600, собрано 700 постов (последний: 2025-11-05)
Обработано offset 700, собрано 800 постов (последний: 2025-10-14)
Обработано offset 800, собрано 900 постов (последний: 2025-09-22)
Обработано offset 900, собрано 1000 постов (последний: 2025-08-22)
Обработано offset 1000, собрано 1100 постов (последний: 2025-07-19)
Обработано offset 1100, собрано 1200 постов (последний: 2025-06-27)
Обработано offset 1200, собрано 1300 постов (последний: 2025-06-06)
Обработано offset 1300, собрано 1

In [ ]:
# щукина
group_id = '-23778827'
name = 'shcukina.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-11)
Обработано offset 100, собрано 200 постов (последний: 2026-03-18)
Обработано offset 200, собрано 300 постов (последний: 2026-02-20)
Обработано offset 300, собрано 400 постов (последний: 2026-01-30)
Обработано offset 400, собрано 500 постов (последний: 2025-12-23)
Обработано offset 500, собрано 600 постов (последний: 2025-11-28)
Обработано offset 600, собрано 700 постов (последний: 2025-11-14)
Обработано offset 700, собрано 800 постов (последний: 2025-10-24)
Обработано offset 800, собрано 900 постов (последний: 2025-09-27)
Обработано offset 900, собрано 1000 постов (последний: 2025-09-05)
Обработано offset 1000, собрано 1100 постов (последний: 2025-08-08)
Обработано offset 1100, собрано 1200 постов (последний: 2025-07-08)
Обработано offset 1200, собрано 1300 постов (последний: 2025-06-15)
Обработано offset 1300, собрано 1

In [ ]:
# менделевыа
group_id = '-31037181'
name = 'muctr.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-01)
Обработано offset 100, собрано 200 постов (последний: 2026-02-25)
Обработано offset 200, собрано 300 постов (последний: 2026-01-23)
Обработано offset 300, собрано 400 постов (последний: 2025-12-11)
Обработано offset 400, собрано 500 постов (последний: 2025-11-11)
Обработано offset 500, собрано 600 постов (последний: 2025-10-05)
Обработано offset 600, собрано 700 постов (последний: 2025-08-28)
Обработано offset 700, собрано 800 постов (последний: 2025-07-21)
Обработано offset 800, собрано 900 постов (последний: 2025-05-31)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-07)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-07)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-03)
Обработано offset 1200, собрано 1300 постов (последний: 2025-01-15)
Обработано offset 1300, собрано 1

In [ ]:
# гнесин
group_id = '-30431394'
name = 'gnesin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-17)
Обработано offset 100, собрано 200 постов (последний: 2025-11-24)
Обработано offset 200, собрано 300 постов (последний: 2025-09-25)
Обработано offset 300, собрано 400 постов (последний: 2025-06-24)
Обработано offset 400, собрано 500 постов (последний: 2025-04-23)
Обработано offset 500, собрано 600 постов (последний: 2025-03-11)
Обработано offset 600, собрано 700 постов (последний: 2024-12-24)
Обработано offset 700, собрано 800 постов (последний: 2024-10-28)
Обработано offset 800, собрано 900 постов (последний: 2024-09-04)
Обработано offset 900, собрано 1000 постов (последний: 2024-05-19)
Обработано offset 1000, собрано 1100 постов (последний: 2024-03-20)
Обработано offset 1100, собрано 1200 постов (последний: 2024-01-25)
Обработано offset 1200, собрано 1300 постов (последний: 2023-11-06)
Обработано offset 1300, собрано 1

In [ ]:
# синергия
group_id = '-38029'
name = 'synergy.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-16)
Обработано offset 100, собрано 200 постов (последний: 2026-01-05)
Обработано offset 200, собрано 300 постов (последний: 2025-11-10)
Обработано offset 300, собрано 400 постов (последний: 2025-08-02)
Обработано offset 400, собрано 500 постов (последний: 2025-05-06)
Обработано offset 500, собрано 600 постов (последний: 2025-04-08)
Обработано offset 600, собрано 700 постов (последний: 2025-03-08)
Обработано offset 700, собрано 800 постов (последний: 2025-02-07)
Обработано offset 800, собрано 900 постов (последний: 2025-01-09)
Обработано offset 900, собрано 1000 постов (последний: 2024-12-05)
Обработано offset 1000, собрано 1100 постов (последний: 2024-11-13)
Обработано offset 1100, собрано 1200 постов (последний: 2024-10-19)
Обработано offset 1200, собрано 1300 постов (последний: 2024-09-28)
Обработано offset 1300, собрано 1

In [ ]:
# политех
group_id = '-33991085'
name = 'polytech.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-03)
Обработано offset 100, собрано 200 постов (последний: 2026-03-04)
Обработано offset 200, собрано 300 постов (последний: 2026-01-26)
Обработано offset 300, собрано 400 постов (последний: 2025-11-14)
Обработано offset 400, собрано 500 постов (последний: 2025-09-23)
Обработано offset 500, собрано 600 постов (последний: 2025-08-12)
Обработано offset 600, собрано 700 постов (последний: 2025-06-27)
Обработано offset 700, собрано 800 постов (последний: 2025-05-06)
Обработано offset 800, собрано 900 постов (последний: 2025-03-22)
Обработано offset 900, собрано 1000 постов (последний: 2025-02-05)
Обработано offset 1000, собрано 1100 постов (последний: 2024-12-06)
Обработано offset 1100, собрано 1200 постов (последний: 2024-10-23)
Обработано offset 1200, собрано 1300 постов (последний: 2024-09-09)
Обработано offset 1300, собрано 1

In [ ]:
# мгппу
group_id = '-38093474'
name = 'mgppu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-10)
Обработано offset 100, собрано 200 постов (последний: 2026-03-20)
Обработано offset 200, собрано 300 постов (последний: 2026-02-27)
Обработано offset 300, собрано 400 постов (последний: 2026-02-09)
Обработано offset 400, собрано 500 постов (последний: 2026-01-19)
Обработано offset 500, собрано 600 постов (последний: 2025-12-12)
Обработано offset 600, собрано 700 постов (последний: 2025-11-14)
Обработано offset 700, собрано 800 постов (последний: 2025-10-22)
Обработано offset 800, собрано 900 постов (последний: 2025-10-01)
Обработано offset 900, собрано 1000 постов (последний: 2025-09-01)
Обработано offset 1000, собрано 1100 постов (последний: 2025-08-04)
Обработано offset 1100, собрано 1200 постов (последний: 2025-07-09)
Обработано offset 1200, собрано 1300 постов (последний: 2025-06-18)
Обработано offset 1300, собрано 1

In [ ]:
# станкин
group_id = '-1332509'
name = 'stankin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-16)
Обработано offset 100, собрано 200 постов (последний: 2026-03-28)
Обработано offset 200, собрано 300 постов (последний: 2026-03-09)
Обработано offset 300, собрано 400 постов (последний: 2026-02-18)
Обработано offset 400, собрано 500 постов (последний: 2026-01-30)
Обработано offset 500, собрано 600 постов (последний: 2026-01-09)
Обработано offset 600, собрано 700 постов (последний: 2025-12-19)
Обработано offset 700, собрано 800 постов (последний: 2025-12-01)
Обработано offset 800, собрано 900 постов (последний: 2025-11-12)
Обработано offset 900, собрано 1000 постов (последний: 2025-10-23)
Обработано offset 1000, собрано 1100 постов (последний: 2025-10-04)
Обработано offset 1100, собрано 1200 постов (последний: 2025-09-14)
Обработано offset 1200, собрано 1300 постов (последний: 2025-08-21)
Обработано offset 1300, собрано 1

In [ ]:
# мгусит
group_id = '-30016632'
name = 'mgusit.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-23)
Обработано offset 100, собрано 200 постов (последний: 2026-04-14)
Обработано offset 200, собрано 300 постов (последний: 2026-04-03)
Обработано offset 300, собрано 400 постов (последний: 2026-03-24)
Обработано offset 400, собрано 500 постов (последний: 2026-03-10)
Обработано offset 500, собрано 600 постов (последний: 2026-02-24)
Обработано offset 600, собрано 700 постов (последний: 2026-02-10)
Обработано offset 700, собрано 800 постов (последний: 2026-01-29)
Обработано offset 800, собрано 900 постов (последний: 2026-01-12)
Обработано offset 900, собрано 1000 постов (последний: 2025-12-23)
Обработано offset 1000, собрано 1100 постов (последний: 2025-12-09)
Обработано offset 1100, собрано 1200 постов (последний: 2025-11-28)
Обработано offset 1200, собрано 1300 постов (последний: 2025-11-17)
Обработано offset 1300, собрано 1

In [ ]:
# ргсу
group_id = '-39683739'
name = 'rgsu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-09)
Обработано offset 100, собрано 200 постов (последний: 2026-03-09)
Обработано offset 200, собрано 300 постов (последний: 2026-02-08)
Обработано offset 300, собрано 400 постов (последний: 2026-01-03)
Обработано offset 400, собрано 500 постов (последний: 2025-12-07)
Обработано offset 500, собрано 600 постов (последний: 2025-11-14)
Обработано offset 600, собрано 700 постов (последний: 2025-10-14)
Обработано offset 700, собрано 800 постов (последний: 2025-09-15)
Обработано offset 800, собрано 900 постов (последний: 2025-08-17)
Обработано offset 900, собрано 1000 постов (последний: 2025-07-10)
Обработано offset 1000, собрано 1100 постов (последний: 2025-06-19)
Обработано offset 1100, собрано 1200 постов (последний: 2025-06-04)
Обработано offset 1200, собрано 1300 постов (последний: 2025-05-13)
Обработано offset 1300, собрано 1

In [ ]:
# росбиотех
group_id = '-168484484'
name = 'rosbiotech.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-20)
Обработано offset 100, собрано 200 постов (последний: 2026-01-27)
Обработано offset 200, собрано 300 постов (последний: 2025-12-05)
Обработано offset 300, собрано 400 постов (последний: 2025-10-24)
Обработано offset 400, собрано 500 постов (последний: 2025-09-15)
Обработано offset 500, собрано 600 постов (последний: 2025-08-02)
Обработано offset 600, собрано 700 постов (последний: 2025-06-16)
Обработано offset 700, собрано 800 постов (последний: 2025-05-05)
Обработано offset 800, собрано 900 постов (последний: 2025-03-24)
Обработано offset 900, собрано 1000 постов (последний: 2025-01-29)
Обработано offset 1000, собрано 1100 постов (последний: 2024-12-09)
Обработано offset 1100, собрано 1200 постов (последний: 2024-11-02)
Обработано offset 1200, собрано 1300 постов (последний: 2024-10-08)
Обработано offset 1300, собрано 1

In [ ]:
# миит
group_id = '-159'
name = 'miit.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-12-25)
Обработано offset 100, собрано 200 постов (последний: 2025-10-15)
Обработано offset 200, собрано 300 постов (последний: 2025-07-07)
Обработано offset 300, собрано 400 постов (последний: 2025-04-17)
Обработано offset 400, собрано 500 постов (последний: 2025-02-03)
Обработано offset 500, собрано 600 постов (последний: 2024-11-20)
Обработано offset 600, собрано 700 постов (последний: 2024-09-20)
Обработано offset 700, собрано 800 постов (последний: 2024-06-20)
Обработано offset 800, собрано 900 постов (последний: 2024-03-15)
Обработано offset 900, собрано 1000 постов (последний: 2023-11-20)
Обработано offset 1000, собрано 1100 постов (последний: 2023-09-07)
Обработано offset 1100, собрано 1200 постов (последний: 2023-05-05)
Обработано offset 1200, собрано 1300 постов (последний: 2023-02-15)
Обработано offset 1300, собрано 1

In [ ]:
# дип академия
group_id = '-181211182'
name = 'dipacademy.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-02)
Обработано offset 100, собрано 200 постов (последний: 2026-02-13)
Обработано offset 200, собрано 300 постов (последний: 2026-01-12)
Обработано offset 300, собрано 400 постов (последний: 2025-12-10)
Обработано offset 400, собрано 500 постов (последний: 2025-11-17)
Обработано offset 500, собрано 600 постов (последний: 2025-10-23)
Обработано offset 600, собрано 700 постов (последний: 2025-10-02)
Обработано offset 700, собрано 800 постов (последний: 2025-09-05)
Обработано offset 800, собрано 900 постов (последний: 2025-07-09)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-10)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-19)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-24)
Обработано offset 1200, собрано 1300 постов (последний: 2025-04-02)
Обработано offset 1300, собрано 1

In [ ]:
# пстгу
group_id = '-97729'
name = 'pstgu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-14)
Обработано offset 100, собрано 200 постов (последний: 2026-01-13)
Обработано offset 200, собрано 300 постов (последний: 2025-11-15)
Обработано offset 300, собрано 400 постов (последний: 2025-09-23)
Обработано offset 400, собрано 500 постов (последний: 2025-07-19)
Обработано offset 500, собрано 600 постов (последний: 2025-05-28)
Обработано offset 600, собрано 700 постов (последний: 2025-04-08)
Обработано offset 700, собрано 800 постов (последний: 2025-02-16)
Обработано offset 800, собрано 900 постов (последний: 2024-12-30)
Обработано offset 900, собрано 1000 постов (последний: 2024-11-20)
Обработано offset 1000, собрано 1100 постов (последний: 2024-10-11)
Обработано offset 1100, собрано 1200 постов (последний: 2024-08-26)
Обработано offset 1200, собрано 1300 постов (последний: 2024-06-19)
Обработано offset 1300, собрано 1

In [ ]:
# мгри
group_id = '-163624433'
name = 'mgri.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-05)
Обработано offset 100, собрано 200 постов (последний: 2026-03-03)
Обработано offset 200, собрано 300 постов (последний: 2026-01-31)
Обработано offset 300, собрано 400 постов (последний: 2025-12-20)
Обработано offset 400, собрано 500 постов (последний: 2025-11-25)
Обработано offset 500, собрано 600 постов (последний: 2025-10-26)
Обработано offset 600, собрано 700 постов (последний: 2025-09-27)
Обработано offset 700, собрано 800 постов (последний: 2025-08-29)
Обработано offset 800, собрано 900 постов (последний: 2025-07-22)
Обработано offset 900, собрано 1000 постов (последний: 2025-06-19)
Обработано offset 1000, собрано 1100 постов (последний: 2025-05-12)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-11)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-15)
Обработано offset 1300, собрано 1

In [ ]:
# балетная академия
group_id = '-211891378'
name = 'balletacademy.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-20)
Обработано offset 100, собрано 200 постов (последний: 2025-09-25)
Обработано offset 200, собрано 300 постов (последний: 2025-04-28)
Обработано offset 300, собрано 400 постов (последний: 2025-01-27)
Обработано offset 400, собрано 500 постов (последний: 2024-11-12)
Обработано offset 500, собрано 600 постов (последний: 2023-05-10)
Обработано offset 600, собрано 684 постов (последний: 2022-03-19)
Постов больше нет
Всего собрано постов: 684
Диапазон дат: 2022-03-19 - 2026-05-05


In [ ]:
# гтсолифк
group_id = '-41423'
name = 'gtsolifk.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-01)
Обработано offset 100, собрано 200 постов (последний: 2026-02-26)
Обработано offset 200, собрано 300 постов (последний: 2026-01-27)
Обработано offset 300, собрано 400 постов (последний: 2025-12-15)
Обработано offset 400, собрано 500 постов (последний: 2025-11-19)
Обработано offset 500, собрано 600 постов (последний: 2025-10-25)
Обработано offset 600, собрано 700 постов (последний: 2025-09-29)
Обработано offset 700, собрано 800 постов (последний: 2025-08-28)
Обработано offset 800, собрано 900 постов (последний: 2025-07-04)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-28)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-29)
Обработано offset 1100, собрано 1200 постов (последний: 2025-04-02)
Обработано offset 1200, собрано 1300 постов (последний: 2025-03-07)
Обработано offset 1300, собрано 1

In [ ]:
# ргхпу
group_id = '-187809308'
name = 'rghpu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-16)
Обработано offset 100, собрано 200 постов (последний: 2026-01-23)
Обработано offset 200, собрано 300 постов (последний: 2025-11-24)
Обработано offset 300, собрано 400 постов (последний: 2025-10-13)
Обработано offset 400, собрано 500 постов (последний: 2025-08-20)
Обработано offset 500, собрано 600 постов (последний: 2025-06-26)
Обработано offset 600, собрано 700 постов (последний: 2025-05-04)
Обработано offset 700, собрано 800 постов (последний: 2025-03-27)
Обработано offset 800, собрано 900 постов (последний: 2025-02-17)
Обработано offset 900, собрано 1000 постов (последний: 2024-12-26)
Обработано offset 1000, собрано 1100 постов (последний: 2024-11-05)
Обработано offset 1100, собрано 1200 постов (последний: 2024-09-13)
Обработано offset 1200, собрано 1300 постов (последний: 2024-07-01)
Обработано offset 1300, собрано 1

In [ ]:
# инсти психоанализа
group_id = '-11290126'
name = 'inpsyho.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-18)
Обработано offset 100, собрано 200 постов (последний: 2025-09-15)
Обработано offset 200, собрано 300 постов (последний: 2025-06-04)
Обработано offset 300, собрано 400 постов (последний: 2025-03-17)
Обработано offset 400, собрано 500 постов (последний: 2024-12-27)
Обработано offset 500, собрано 600 постов (последний: 2024-10-10)
Обработано offset 600, собрано 700 постов (последний: 2024-06-12)
Обработано offset 700, собрано 800 постов (последний: 2024-02-27)
Обработано offset 800, собрано 900 постов (последний: 2023-12-19)
Обработано offset 900, собрано 1000 постов (последний: 2023-10-20)
Обработано offset 1000, собрано 1100 постов (последний: 2023-08-29)
Обработано offset 1100, собрано 1200 постов (последний: 2023-06-13)
Обработано offset 1200, собрано 1300 постов (последний: 2023-04-05)
Обработано offset 1300, собрано 1

In [ ]:
# мглу
group_id = '-214354323'
name = 'mslu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-16)
Обработано offset 100, собрано 200 постов (последний: 2026-02-09)
Обработано offset 200, собрано 300 постов (последний: 2025-12-18)
Обработано offset 300, собрано 400 постов (последний: 2025-11-02)
Обработано offset 400, собрано 500 постов (последний: 2025-09-10)
Обработано offset 500, собрано 600 постов (последний: 2025-06-27)
Обработано offset 600, собрано 700 постов (последний: 2025-05-15)
Обработано offset 700, собрано 800 постов (последний: 2025-03-28)
Обработано offset 800, собрано 900 постов (последний: 2025-02-03)
Обработано offset 900, собрано 931 постов (последний: 2023-11-06)
Постов больше нет
Всего собрано постов: 931
Диапазон дат: 2023-11-06 - 2026-05-06


In [ ]:
# мадм
group_id = '-138541024'
name = 'madi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-08)
Обработано offset 100, собрано 200 постов (последний: 2025-12-12)
Обработано offset 200, собрано 300 постов (последний: 2025-07-24)
Обработано offset 300, собрано 400 постов (последний: 2025-03-24)
Обработано offset 400, собрано 500 постов (последний: 2024-11-23)
Обработано offset 500, собрано 600 постов (последний: 2024-06-03)
Обработано offset 600, собрано 700 постов (последний: 2024-03-07)
Обработано offset 700, собрано 800 постов (последний: 2023-11-08)
Обработано offset 800, собрано 900 постов (последний: 2023-07-05)
Обработано offset 900, собрано 1000 постов (последний: 2023-04-01)
Обработано offset 1000, собрано 1100 постов (последний: 2023-01-25)
Обработано offset 1100, собрано 1200 постов (последний: 2022-11-14)
Обработано offset 1200, собрано 1300 постов (последний: 2022-09-13)
Обработано offset 1300, собрано 1

In [ ]:
# скрябина
group_id = '-103477801'
name = 'mgavm.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-31)
Обработано offset 100, собрано 200 постов (последний: 2026-02-17)
Обработано offset 200, собрано 300 постов (последний: 2026-01-12)
Обработано offset 300, собрано 400 постов (последний: 2025-11-21)
Обработано offset 400, собрано 500 постов (последний: 2025-10-20)
Обработано offset 500, собрано 600 постов (последний: 2025-09-16)
Обработано offset 600, собрано 700 постов (последний: 2025-07-23)
Обработано offset 700, собрано 800 постов (последний: 2025-06-03)
Обработано offset 800, собрано 900 постов (последний: 2025-04-16)
Обработано offset 900, собрано 1000 постов (последний: 2025-03-05)
Обработано offset 1000, собрано 1100 постов (последний: 2025-01-31)
Обработано offset 1100, собрано 1200 постов (последний: 2024-12-16)
Обработано offset 1200, собрано 1300 постов (последний: 2024-11-15)
Обработано offset 1300, собрано 1

In [ ]:
# консерватория
group_id = '-36100537'
name = 'mocsons.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-10-20)
Обработано offset 100, собрано 200 постов (последний: 2025-04-02)
Обработано offset 200, собрано 300 постов (последний: 2025-02-16)
Обработано offset 300, собрано 400 постов (последний: 2024-12-17)
Обработано offset 400, собрано 500 постов (последний: 2024-10-30)
Обработано offset 500, собрано 600 постов (последний: 2024-09-16)
Обработано offset 600, собрано 700 постов (последний: 2024-07-04)
Обработано offset 700, собрано 800 постов (последний: 2024-04-29)
Обработано offset 800, собрано 900 постов (последний: 2024-03-05)
Обработано offset 900, собрано 1000 постов (последний: 2024-01-18)
Обработано offset 1000, собрано 1100 постов (последний: 2023-11-02)
Обработано offset 1100, собрано 1200 постов (последний: 2023-09-14)
Обработано offset 1200, собрано 1300 постов (последний: 2023-06-18)
Обработано offset 1300, собрано 1

In [ ]:
# евдокимова
group_id = '-46611574'
name = 'msmsu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-25)
Обработано offset 100, собрано 200 постов (последний: 2025-10-23)
Обработано offset 200, собрано 300 постов (последний: 2025-08-10)
Обработано offset 300, собрано 400 постов (последний: 2025-05-08)
Обработано offset 400, собрано 500 постов (последний: 2025-03-13)
Обработано offset 500, собрано 600 постов (последний: 2025-01-09)
Обработано offset 600, собрано 700 постов (последний: 2024-11-14)
Обработано offset 700, собрано 800 постов (последний: 2024-09-06)
Обработано offset 800, собрано 900 постов (последний: 2024-05-17)
Обработано offset 900, собрано 1000 постов (последний: 2024-02-10)
Обработано offset 1000, собрано 1100 постов (последний: 2023-11-29)
Обработано offset 1100, собрано 1200 постов (последний: 2023-10-16)
Обработано offset 1200, собрано 1300 постов (последний: 2023-06-22)
Обработано offset 1300, собрано 1

In [ ]:
# рэш
group_id = '-471300'
name = 'rash.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-20)
Обработано offset 100, собрано 200 постов (последний: 2026-01-29)
Обработано offset 200, собрано 300 постов (последний: 2025-11-27)
Обработано offset 300, собрано 400 постов (последний: 2025-10-09)
Обработано offset 400, собрано 500 постов (последний: 2025-08-18)
Обработано offset 500, собрано 600 постов (последний: 2025-06-30)
Обработано offset 600, собрано 700 постов (последний: 2025-05-12)
Обработано offset 700, собрано 800 постов (последний: 2025-03-21)
Обработано offset 800, собрано 900 постов (последний: 2025-01-29)
Обработано offset 900, собрано 1000 постов (последний: 2024-11-27)
Обработано offset 1000, собрано 1100 постов (последний: 2024-10-10)
Обработано offset 1100, собрано 1200 постов (последний: 2024-08-02)
Обработано offset 1200, собрано 1300 постов (последний: 2024-05-17)
Обработано offset 1300, собрано 1

In [ ]:
# мии гаик
group_id = '-44133092'
name = 'miigaik.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-17)
Обработано offset 100, собрано 200 постов (последний: 2026-03-31)
Обработано offset 200, собрано 300 постов (последний: 2026-03-13)
Обработано offset 300, собрано 400 постов (последний: 2026-02-26)
Обработано offset 400, собрано 500 постов (последний: 2026-02-08)
Обработано offset 500, собрано 600 постов (последний: 2025-12-31)
Обработано offset 600, собрано 700 постов (последний: 2025-12-10)
Обработано offset 700, собрано 800 постов (последний: 2025-11-25)
Обработано offset 800, собрано 900 постов (последний: 2025-11-02)
Обработано offset 900, собрано 1000 постов (последний: 2025-09-15)
Обработано offset 1000, собрано 1100 постов (последний: 2025-07-29)
Обработано offset 1100, собрано 1200 постов (последний: 2025-06-10)
Обработано offset 1200, собрано 1300 постов (последний: 2025-05-05)
Обработано offset 1300, собрано 1

In [ ]:
# мфуа
group_id = '-4292'
name = 'mfua.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-09-03)
Обработано offset 100, собрано 200 постов (последний: 2024-12-06)
Обработано offset 200, собрано 300 постов (последний: 2024-05-07)
Обработано offset 300, собрано 400 постов (последний: 2023-11-23)
Обработано offset 400, собрано 500 постов (последний: 2023-06-19)
Обработано offset 500, собрано 600 постов (последний: 2023-02-15)
Обработано offset 600, собрано 700 постов (последний: 2022-11-22)
Обработано offset 700, собрано 800 постов (последний: 2022-09-09)
Обработано offset 800, собрано 900 постов (последний: 2022-06-05)
Обработано offset 900, собрано 1000 постов (последний: 2022-03-29)
Обработано offset 1000, собрано 1100 постов (последний: 2021-12-22)
Обработано offset 1100, собрано 1200 постов (последний: 2021-09-10)
Обработано offset 1200, собрано 1300 постов (последний: 2021-04-19)
Обработано offset 1300, собрано 1

In [ ]:
# мархи
group_id = '-203483546'
name = 'marhi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-13)
Обработано offset 100, собрано 200 постов (последний: 2025-10-03)
Обработано offset 200, собрано 300 постов (последний: 2025-07-16)
Обработано offset 300, собрано 400 постов (последний: 2025-04-28)
Обработано offset 400, собрано 500 постов (последний: 2025-02-28)
Обработано offset 500, собрано 600 постов (последний: 2024-12-03)
Обработано offset 600, собрано 700 постов (последний: 2024-09-26)
Обработано offset 700, собрано 800 постов (последний: 2024-01-24)
Обработано offset 800, собрано 900 постов (последний: 2023-10-02)
Обработано offset 900, собрано 1000 постов (последний: 2023-05-17)
Обработано offset 1000, собрано 1100 постов (последний: 2023-02-14)
Обработано offset 1100, собрано 1200 постов (последний: 2022-11-18)
Обработано offset 1200, собрано 1300 постов (последний: 2022-09-22)
Обработано offset 1300, собрано 1

In [ ]:
# пушкина
group_id = '-30164925'
name = 'pushkin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-17)
Обработано offset 100, собрано 200 постов (последний: 2025-11-19)
Обработано offset 200, собрано 300 постов (последний: 2025-09-08)
Обработано offset 300, собрано 400 постов (последний: 2025-07-09)
Обработано offset 400, собрано 500 постов (последний: 2025-05-21)
Обработано offset 500, собрано 600 постов (последний: 2025-03-10)
Обработано offset 600, собрано 700 постов (последний: 2024-12-13)
Обработано offset 700, собрано 800 постов (последний: 2024-10-28)
Обработано offset 800, собрано 900 постов (последний: 2024-09-16)
Обработано offset 900, собрано 1000 постов (последний: 2024-07-29)
Обработано offset 1000, собрано 1100 постов (последний: 2024-06-14)
Обработано offset 1100, собрано 1200 постов (последний: 2024-05-24)
Обработано offset 1200, собрано 1300 постов (последний: 2024-04-30)
Обработано offset 1300, собрано 1

In [ ]:
# мхат
group_id = '-1010597'
name = 'mhat.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-01-10)
Обработано offset 100, собрано 200 постов (последний: 2024-06-28)
Обработано offset 200, собрано 300 постов (последний: 2024-03-30)
Обработано offset 300, собрано 400 постов (последний: 2024-01-19)
Обработано offset 400, собрано 500 постов (последний: 2023-10-27)
Обработано offset 500, собрано 600 постов (последний: 2023-09-08)
Обработано offset 600, собрано 700 постов (последний: 2023-06-11)
Обработано offset 700, собрано 800 постов (последний: 2023-04-13)
Обработано offset 800, собрано 900 постов (последний: 2023-02-28)
Обработано offset 900, собрано 1000 постов (последний: 2022-12-08)
Обработано offset 1000, собрано 1100 постов (последний: 2022-09-21)
Обработано offset 1100, собрано 1200 постов (последний: 2022-06-13)
Обработано offset 1200, собрано 1300 постов (последний: 2022-03-27)
Обработано offset 1300, собрано 1

In [ ]:
# мгутм
group_id = '-94240017'
name = 'mgutm.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-31)
Обработано offset 100, собрано 200 постов (последний: 2026-02-11)
Обработано offset 200, собрано 300 постов (последний: 2025-12-04)
Обработано offset 300, собрано 400 постов (последний: 2025-10-15)
Обработано offset 400, собрано 500 постов (последний: 2025-08-12)
Обработано offset 500, собрано 600 постов (последний: 2025-05-02)
Обработано offset 600, собрано 700 постов (последний: 2025-02-05)
Обработано offset 700, собрано 800 постов (последний: 2024-10-21)
Обработано offset 800, собрано 900 постов (последний: 2024-07-25)
Обработано offset 900, собрано 1000 постов (последний: 2024-05-15)
Обработано offset 1000, собрано 1100 постов (последний: 2024-02-21)
Обработано offset 1100, собрано 1200 постов (последний: 2023-11-22)
Обработано offset 1200, собрано 1300 постов (последний: 2023-09-19)
Обработано offset 1300, собрано 1

In [ ]:
# мтуси
group_id = '-786'
name = 'mtusi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-12)
Обработано offset 100, собрано 200 постов (последний: 2025-10-14)
Обработано offset 200, собрано 300 постов (последний: 2025-06-06)
Обработано offset 300, собрано 400 постов (последний: 2025-04-30)
Обработано offset 400, собрано 500 постов (последний: 2025-03-28)
Обработано offset 500, собрано 600 постов (последний: 2025-02-19)
Обработано offset 600, собрано 700 постов (последний: 2024-12-31)
Обработано offset 700, собрано 800 постов (последний: 2024-11-27)
Обработано offset 800, собрано 900 постов (последний: 2024-11-02)
Обработано offset 900, собрано 1000 постов (последний: 2024-10-02)
Обработано offset 1000, собрано 1100 постов (последний: 2024-09-02)
Обработано offset 1100, собрано 1200 постов (последний: 2024-06-29)
Обработано offset 1200, собрано 1300 постов (последний: 2024-05-21)
Обработано offset 1300, собрано 1

In [ ]:
# гуз
group_id = '-8990'
name = 'guz.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-07)
Обработано offset 100, собрано 200 постов (последний: 2026-03-06)
Обработано offset 200, собрано 300 постов (последний: 2026-01-27)
Обработано offset 300, собрано 400 постов (последний: 2025-12-10)
Обработано offset 400, собрано 500 постов (последний: 2025-11-17)
Обработано offset 500, собрано 600 постов (последний: 2025-10-15)
Обработано offset 600, собрано 700 постов (последний: 2025-09-10)
Обработано offset 700, собрано 800 постов (последний: 2025-07-11)
Обработано offset 800, собрано 900 постов (последний: 2025-05-27)
Обработано offset 900, собрано 1000 постов (последний: 2025-04-20)
Обработано offset 1000, собрано 1100 постов (последний: 2025-03-24)
Обработано offset 1100, собрано 1200 постов (последний: 2025-02-25)
Обработано offset 1200, собрано 1300 постов (последний: 2025-01-21)
Обработано offset 1300, собрано 1

In [ ]:
# гитр
group_id = '-32851'
name = 'gitr.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-13)
Обработано offset 100, собрано 200 постов (последний: 2025-12-31)
Обработано offset 200, собрано 300 постов (последний: 2025-10-30)
Обработано offset 300, собрано 400 постов (последний: 2025-08-30)
Обработано offset 400, собрано 500 постов (последний: 2025-05-26)
Обработано offset 500, собрано 600 постов (последний: 2025-04-12)
Обработано offset 600, собрано 700 постов (последний: 2025-02-10)
Обработано offset 700, собрано 800 постов (последний: 2024-12-21)
Обработано offset 800, собрано 900 постов (последний: 2024-11-13)
Обработано offset 900, собрано 1000 постов (последний: 2024-07-22)
Обработано offset 1000, собрано 1100 постов (последний: 2024-05-02)
Обработано offset 1100, собрано 1200 постов (последний: 2024-03-07)
Обработано offset 1200, собрано 1300 постов (последний: 2023-12-21)
Обработано offset 1300, собрано 1

In [ ]:
# вавт
group_id = '-194691927'
name = 'vavt.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-24)
Обработано offset 100, собрано 200 постов (последний: 2026-01-25)
Обработано offset 200, собрано 300 постов (последний: 2025-12-01)
Обработано offset 300, собрано 400 постов (последний: 2025-10-06)
Обработано offset 400, собрано 500 постов (последний: 2025-07-30)
Обработано offset 500, собрано 600 постов (последний: 2025-05-26)
Обработано offset 600, собрано 700 постов (последний: 2025-04-10)
Обработано offset 700, собрано 800 постов (последний: 2025-02-09)
Обработано offset 800, собрано 900 постов (последний: 2024-11-29)
Обработано offset 900, собрано 1000 постов (последний: 2024-10-14)
Обработано offset 1000, собрано 1100 постов (последний: 2024-08-13)
Обработано offset 1100, собрано 1200 постов (последний: 2024-06-06)
Обработано offset 1200, собрано 1300 постов (последний: 2024-03-09)
Обработано offset 1300, собрано 1

In [ ]:
# гаугн
group_id = '-37520'
name = 'gaugn.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-04-06)
Обработано offset 100, собрано 200 постов (последний: 2026-02-27)
Обработано offset 200, собрано 300 постов (последний: 2026-01-20)
Обработано offset 300, собрано 400 постов (последний: 2025-12-05)
Обработано offset 400, собрано 500 постов (последний: 2025-11-07)
Обработано offset 500, собрано 600 постов (последний: 2025-09-29)
Обработано offset 600, собрано 700 постов (последний: 2025-08-18)
Обработано offset 700, собрано 800 постов (последний: 2025-07-10)
Обработано offset 800, собрано 900 постов (последний: 2025-06-11)
Обработано offset 900, собрано 1000 постов (последний: 2025-05-01)
Обработано offset 1000, собрано 1100 постов (последний: 2025-04-04)
Обработано offset 1100, собрано 1200 постов (последний: 2025-03-08)
Обработано offset 1200, собрано 1300 постов (последний: 2025-02-12)
Обработано offset 1300, собрано 1

In [ ]:
# ипполитовка
group_id = '-160773003'
name = 'ippolit.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-22)
Обработано offset 100, собрано 200 постов (последний: 2026-01-25)
Обработано offset 200, собрано 300 постов (последний: 2025-11-30)
Обработано offset 300, собрано 400 постов (последний: 2025-10-21)
Обработано offset 400, собрано 500 постов (последний: 2025-08-28)
Обработано offset 500, собрано 600 постов (последний: 2025-05-23)
Обработано offset 600, собрано 700 постов (последний: 2025-04-18)
Обработано offset 700, собрано 800 постов (последний: 2025-03-18)
Обработано offset 800, собрано 900 постов (последний: 2025-02-04)
Обработано offset 900, собрано 1000 постов (последний: 2024-12-14)
Обработано offset 1000, собрано 1100 постов (последний: 2024-11-14)
Обработано offset 1100, собрано 1200 постов (последний: 2024-10-10)
Обработано offset 1200, собрано 1300 постов (последний: 2024-07-29)
Обработано offset 1300, собрано 1

In [ ]:
# rosnou
group_id = '-152003'
name = 'rosnou.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-11-12)
Обработано offset 100, собрано 200 постов (последний: 2025-06-24)
Обработано offset 200, собрано 300 постов (последний: 2024-12-11)
Обработано offset 300, собрано 400 постов (последний: 2024-07-24)
Обработано offset 400, собрано 500 постов (последний: 2024-02-09)
Обработано offset 500, собрано 600 постов (последний: 2023-09-27)
Обработано offset 600, собрано 700 постов (последний: 2023-06-09)
Обработано offset 700, собрано 800 постов (последний: 2023-02-27)
Обработано offset 800, собрано 900 постов (последний: 2022-11-24)
Обработано offset 900, собрано 1000 постов (последний: 2022-09-06)
Обработано offset 1000, собрано 1100 постов (последний: 2022-05-26)
Обработано offset 1100, собрано 1200 постов (последний: 2022-02-10)
Обработано offset 1200, собрано 1300 постов (последний: 2021-10-01)
Обработано offset 1300, собрано 1

In [ ]:
# андриак
group_id = '-35003319'
name = 'andriaka.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-20)
Обработано offset 100, собрано 200 постов (последний: 2025-11-05)
Обработано offset 200, собрано 300 постов (последний: 2025-06-16)
Обработано offset 300, собрано 400 постов (последний: 2025-02-21)
Обработано offset 400, собрано 500 постов (последний: 2024-10-11)
Обработано offset 500, собрано 600 постов (последний: 2024-06-06)
Обработано offset 600, собрано 700 постов (последний: 2024-03-08)
Обработано offset 700, собрано 800 постов (последний: 2023-10-16)
Обработано offset 800, собрано 900 постов (последний: 2023-05-15)
Обработано offset 900, собрано 1000 постов (последний: 2023-01-19)
Обработано offset 1000, собрано 1100 постов (последний: 2022-10-09)
Обработано offset 1100, собрано 1200 постов (последний: 2022-07-08)
Обработано offset 1200, собрано 1300 постов (последний: 2022-04-20)
Обработано offset 1300, собрано 1

In [ ]:
# иси
group_id = '-26099662'
name = 'isi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-24)
Обработано offset 100, собрано 200 постов (последний: 2025-12-20)
Обработано offset 200, собрано 300 постов (последний: 2025-09-28)
Обработано offset 300, собрано 400 постов (последний: 2025-07-19)
Обработано offset 400, собрано 500 постов (последний: 2025-06-24)
Обработано offset 500, собрано 600 постов (последний: 2025-05-27)
Обработано offset 600, собрано 700 постов (последний: 2025-05-03)
Обработано offset 700, собрано 800 постов (последний: 2025-04-14)
Обработано offset 800, собрано 900 постов (последний: 2025-03-21)
Обработано offset 900, собрано 1000 постов (последний: 2025-02-26)
Обработано offset 1000, собрано 1100 постов (последний: 2025-02-01)
Обработано offset 1100, собрано 1200 постов (последний: 2025-01-03)
Обработано offset 1200, собрано 1300 постов (последний: 2024-12-13)
Обработано offset 1300, собрано 1

In [ ]:
# имес
group_id = '-104797'
name = 'imes.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-08)
Обработано offset 100, собрано 200 постов (последний: 2025-10-01)
Обработано offset 200, собрано 300 постов (последний: 2025-06-02)
Обработано offset 300, собрано 400 постов (последний: 2025-04-03)
Обработано offset 400, собрано 500 постов (последний: 2025-02-12)
Обработано offset 500, собрано 600 постов (последний: 2024-12-03)
Обработано offset 600, собрано 700 постов (последний: 2024-10-07)
Обработано offset 700, собрано 800 постов (последний: 2024-07-12)
Обработано offset 800, собрано 900 постов (последний: 2024-05-07)
Обработано offset 900, собрано 1000 постов (последний: 2024-01-25)
Обработано offset 1000, собрано 1100 постов (последний: 2023-10-17)
Обработано offset 1100, собрано 1200 постов (последний: 2023-03-14)
Обработано offset 1200, собрано 1300 постов (последний: 2022-11-21)
Обработано offset 1300, собрано 1

In [ ]:
# щепка
group_id = '-7555704'
name = 'shepka.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-03)
Обработано offset 100, собрано 200 постов (последний: 2025-12-24)
Обработано offset 200, собрано 300 постов (последний: 2025-10-26)
Обработано offset 300, собрано 400 постов (последний: 2025-08-08)
Обработано offset 400, собрано 500 постов (последний: 2025-05-30)
Обработано offset 500, собрано 600 постов (последний: 2025-04-12)
Обработано offset 600, собрано 700 постов (последний: 2025-02-09)
Обработано offset 700, собрано 800 постов (последний: 2024-12-04)
Обработано offset 800, собрано 900 постов (последний: 2024-10-24)
Обработано offset 900, собрано 1000 постов (последний: 2024-09-11)
Обработано offset 1000, собрано 1100 постов (последний: 2024-07-27)
Обработано offset 1100, собрано 1200 постов (последний: 2024-06-12)
Обработано offset 1200, собрано 1300 постов (последний: 2024-05-09)
Обработано offset 1300, собрано 1

In [ ]:
# ииле
group_id = '-33044983'
name = 'iile.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-04)
Обработано offset 100, собрано 200 постов (последний: 2026-01-13)
Обработано offset 200, собрано 300 постов (последний: 2025-10-20)
Обработано offset 300, собрано 400 постов (последний: 2025-06-20)
Обработано offset 400, собрано 500 постов (последний: 2025-04-04)
Обработано offset 500, собрано 600 постов (последний: 2025-01-18)
Обработано offset 600, собрано 700 постов (последний: 2024-10-25)
Обработано offset 700, собрано 800 постов (последний: 2024-06-17)
Обработано offset 800, собрано 900 постов (последний: 2024-02-19)
Обработано offset 900, собрано 1000 постов (последний: 2023-12-20)
Обработано offset 1000, собрано 1100 постов (последний: 2023-11-12)
Обработано offset 1100, собрано 1200 постов (последний: 2023-09-16)
Обработано offset 1200, собрано 1300 постов (последний: 2023-06-28)
Обработано offset 1300, собрано 1

In [ ]:
# мос ити
group_id = '-71768146'
name = 'mositi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-26)
Обработано offset 100, собрано 200 постов (последний: 2026-01-30)
Обработано offset 200, собрано 300 постов (последний: 2025-12-05)
Обработано offset 300, собрано 400 постов (последний: 2025-10-17)
Обработано offset 400, собрано 500 постов (последний: 2025-09-20)
Обработано offset 500, собрано 600 постов (последний: 2025-07-26)
Обработано offset 600, собрано 700 постов (последний: 2025-06-17)
Обработано offset 700, собрано 800 постов (последний: 2025-04-24)
Обработано offset 800, собрано 900 постов (последний: 2025-02-16)
Обработано offset 900, собрано 1000 постов (последний: 2024-11-29)
Обработано offset 1000, собрано 1100 постов (последний: 2024-09-11)
Обработано offset 1100, собрано 1200 постов (последний: 2024-06-04)
Обработано offset 1200, собрано 1300 постов (последний: 2023-11-22)
Обработано offset 1300, собрано 1

In [ ]:
# мосгу
group_id = '-4456'
name = 'mosgu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-18)
Обработано offset 100, собрано 200 постов (последний: 2026-01-20)
Обработано offset 200, собрано 300 постов (последний: 2025-11-21)
Обработано offset 300, собрано 400 постов (последний: 2025-09-29)
Обработано offset 400, собрано 500 постов (последний: 2025-07-22)
Обработано offset 500, собрано 600 постов (последний: 2025-05-15)
Обработано offset 600, собрано 700 постов (последний: 2025-03-04)
Обработано offset 700, собрано 800 постов (последний: 2024-11-25)
Обработано offset 800, собрано 900 постов (последний: 2024-08-27)
Обработано offset 900, собрано 1000 постов (последний: 2024-05-11)
Обработано offset 1000, собрано 1100 постов (последний: 2023-12-04)
Обработано offset 1100, собрано 1200 постов (последний: 2023-09-02)
Обработано offset 1200, собрано 1300 постов (последний: 2023-06-09)
Обработано offset 1300, собрано 1

In [ ]:
# витте
group_id = '-58519483'
name = 'vitte.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-13)
Обработано offset 100, собрано 200 постов (последний: 2025-09-15)
Обработано offset 200, собрано 300 постов (последний: 2025-06-12)
Обработано offset 300, собрано 400 постов (последний: 2025-02-17)
Обработано offset 400, собрано 500 постов (последний: 2024-10-30)
Обработано offset 500, собрано 600 постов (последний: 2024-08-27)
Обработано offset 600, собрано 700 постов (последний: 2024-05-24)
Обработано offset 700, собрано 800 постов (последний: 2023-12-28)
Обработано offset 800, собрано 900 постов (последний: 2023-09-05)
Обработано offset 900, собрано 1000 постов (последний: 2023-05-03)
Обработано offset 1000, собрано 1100 постов (последний: 2023-02-20)
Обработано offset 1100, собрано 1200 постов (последний: 2022-11-07)
Обработано offset 1200, собрано 1300 постов (последний: 2022-09-05)
Обработано offset 1300, собрано 1

In [ ]:
# юстиции
group_id = '-111837730'
name = 'rpamu.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2024-01-11)
Обработано offset 100, собрано 200 постов (последний: 2022-11-17)
Обработано offset 200, собрано 300 постов (последний: 2021-05-08)
Обработано offset 300, собрано 400 постов (последний: 2020-10-14)
Обработано offset 400, собрано 500 постов (последний: 2020-04-28)

- Достигли 2019 года -
Последний пост: 3312 от 2019-12-30
Всего собрано постов: 598
Диапазон дат: 2020-01-09 - 2026-05-01


In [ ]:
# mhpidesign
group_id = '-29550550'
name = 'mhpidesign.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-10-17)
Обработано offset 100, собрано 200 постов (последний: 2025-07-28)
Обработано offset 200, собрано 300 постов (последний: 2025-05-14)
Обработано offset 300, собрано 400 постов (последний: 2025-02-09)
Обработано offset 400, собрано 500 постов (последний: 2024-10-24)
Обработано offset 500, собрано 600 постов (последний: 2024-07-16)
Обработано offset 600, собрано 700 постов (последний: 2024-03-17)
Обработано offset 700, собрано 800 постов (последний: 2023-10-13)
Обработано offset 800, собрано 900 постов (последний: 2023-01-10)
Обработано offset 900, собрано 1000 постов (последний: 2022-06-02)
Обработано offset 1000, собрано 1100 постов (последний: 2021-05-29)
Обработано offset 1100, собрано 1200 постов (последний: 2020-04-23)

- Достигли 2019 года -
Последний пост: 1921 от 2019-12-26
Всего собрано постов: 1236
Диапазон дат: 2

In [ ]:
# суриков
group_id = '-194894715'
name = 'surikov.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 21 постов (последний: 2024-12-19)
Постов больше нет
Всего собрано постов: 21
Диапазон дат: 2024-12-19 - 2026-03-24


In [ ]:
# жирик
group_id = '-19923410'
name = 'uwc.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-21)
Обработано offset 100, собрано 200 постов (последний: 2025-11-30)
Обработано offset 200, собрано 300 постов (последний: 2025-09-25)
Обработано offset 300, собрано 400 постов (последний: 2025-06-16)
Обработано offset 400, собрано 500 постов (последний: 2025-03-12)
Обработано offset 500, собрано 600 постов (последний: 2024-10-15)
Обработано offset 600, собрано 700 постов (последний: 2024-05-20)
Обработано offset 700, собрано 800 постов (последний: 2024-01-24)
Обработано offset 800, собрано 900 постов (последний: 2023-10-16)
Обработано offset 900, собрано 1000 постов (последний: 2023-07-20)
Обработано offset 1000, собрано 1100 постов (последний: 2023-04-27)
Обработано offset 1100, собрано 1200 постов (последний: 2023-02-15)
Обработано offset 1200, собрано 1300 постов (последний: 2022-11-11)
Обработано offset 1300, собрано 1

In [ ]:
# rgupru
group_id = '-212123803'
name = 'rgupru.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2024-08-28)
Обработано offset 100, собрано 200 постов (последний: 2024-04-24)
Обработано offset 200, собрано 300 постов (последний: 2023-08-10)
Обработано offset 300, собрано 400 постов (последний: 2022-12-26)
Обработано offset 400, собрано 500 постов (последний: 2022-05-21)
Обработано offset 500, собрано 517 постов (последний: 2022-04-08)
Постов больше нет
Всего собрано постов: 517
Диапазон дат: 2022-04-08 - 2026-05-05


In [ ]:
# ммамос
group_id = '-84480950'
name = 'mmamos.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-07)
Обработано offset 100, собрано 200 постов (последний: 2025-12-10)
Обработано offset 200, собрано 300 постов (последний: 2025-09-26)
Обработано offset 300, собрано 400 постов (последний: 2025-06-25)
Обработано offset 400, собрано 500 постов (последний: 2025-04-21)
Обработано offset 500, собрано 600 постов (последний: 2025-03-02)
Обработано offset 600, собрано 700 постов (последний: 2024-12-27)
Обработано offset 700, собрано 800 постов (последний: 2024-11-16)
Обработано offset 800, собрано 900 постов (последний: 2024-09-30)
Обработано offset 900, собрано 1000 постов (последний: 2024-05-29)
Обработано offset 1000, собрано 1100 постов (последний: 2024-03-19)
Обработано offset 1100, собрано 1200 постов (последний: 2023-12-25)
Обработано offset 1200, собрано 1300 постов (последний: 2023-10-19)
Обработано offset 1300, собрано 1

In [ ]:
# miuniversity
group_id = '-9003905'
name = 'miuniversity.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-09-16)
Обработано offset 100, собрано 200 постов (последний: 2024-12-12)
Обработано offset 200, собрано 300 постов (последний: 2024-06-03)
Обработано offset 300, собрано 400 постов (последний: 2023-09-06)
Обработано offset 400, собрано 500 постов (последний: 2022-11-22)
Обработано offset 500, собрано 600 постов (последний: 2022-05-11)
Обработано offset 600, собрано 700 постов (последний: 2021-04-09)
Обработано offset 700, собрано 800 постов (последний: 2020-09-17)

- Достигли 2019 года -
Последний пост: 9133 от 2019-12-16
Всего собрано постов: 836
Диапазон дат: 2020-02-13 - 2026-05-06


In [ ]:
# муии
group_id = '-32026522'
name = 'mui.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-12-13)
Обработано offset 100, собрано 200 постов (последний: 2025-06-11)
Обработано offset 200, собрано 300 постов (последний: 2025-02-11)
Обработано offset 300, собрано 400 постов (последний: 2024-05-08)
Обработано offset 400, собрано 500 постов (последний: 2023-10-03)
Обработано offset 500, собрано 600 постов (последний: 2023-01-24)
Обработано offset 600, собрано 700 постов (последний: 2022-07-28)
Обработано offset 700, собрано 800 постов (последний: 2021-10-13)
Обработано offset 800, собрано 900 постов (последний: 2020-12-09)
Обработано offset 900, собрано 1000 постов (последний: 2020-01-02)

- Достигли 2019 года -
Последний пост: 4329 от 2019-12-31
Всего собрано постов: 1000
Диапазон дат: 2020-01-02 - 2026-05-05


In [ ]:
# шанинка
group_id = '-82402911'
name = 'shanin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-10-14)
Обработано offset 100, собрано 200 постов (последний: 2025-04-04)
Обработано offset 200, собрано 300 постов (последний: 2024-11-22)
Обработано offset 300, собрано 400 постов (последний: 2024-07-29)
Обработано offset 400, собрано 500 постов (последний: 2024-04-18)
Обработано offset 500, собрано 600 постов (последний: 2024-01-18)
Обработано offset 600, собрано 700 постов (последний: 2023-09-25)
Обработано offset 700, собрано 800 постов (последний: 2023-05-27)
Обработано offset 800, собрано 900 постов (последний: 2023-02-04)
Обработано offset 900, собрано 1000 постов (последний: 2022-10-19)
Обработано offset 1000, собрано 1100 постов (последний: 2022-06-27)
Обработано offset 1100, собрано 1200 постов (последний: 2022-03-29)
Обработано offset 1200, собрано 1300 постов (последний: 2021-12-15)
Обработано offset 1300, собрано 1

In [ ]:
# райкин
group_id = '-59762666'
name = 'raikin.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-04-21)
Обработано offset 100, собрано 200 постов (последний: 2024-10-29)
Обработано offset 200, собрано 300 постов (последний: 2024-02-27)
Обработано offset 300, собрано 400 постов (последний: 2023-10-06)
Обработано offset 400, собрано 500 постов (последний: 2023-02-17)
Обработано offset 500, собрано 600 постов (последний: 2022-10-29)
Обработано offset 600, собрано 700 постов (последний: 2021-03-19)

- Достигли 2019 года -
Последний пост: 963 от 2019-12-27
Всего собрано постов: 786
Диапазон дат: 2020-01-13 - 2026-05-05


In [ ]:
# миту маси
group_id = '-29193864'
name = 'mitumasi.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-02-27)
Обработано offset 100, собрано 200 постов (последний: 2025-12-12)
Обработано offset 200, собрано 300 постов (последний: 2025-09-25)
Обработано offset 300, собрано 400 постов (последний: 2025-07-30)
Обработано offset 400, собрано 500 постов (последний: 2025-06-20)
Обработано offset 500, собрано 600 постов (последний: 2025-05-21)
Обработано offset 600, собрано 700 постов (последний: 2025-04-29)
Обработано offset 700, собрано 800 постов (последний: 2025-04-01)
Обработано offset 800, собрано 900 постов (последний: 2025-02-27)
Обработано offset 900, собрано 1000 постов (последний: 2025-01-25)
Обработано offset 1000, собрано 1100 постов (последний: 2024-12-17)
Обработано offset 1100, собрано 1200 постов (последний: 2024-11-14)
Обработано offset 1200, собрано 1300 постов (последний: 2024-09-12)
Обработано offset 1300, собрано 1

In [ ]:
# мстука и
group_id = '-45732330'
name = 'mstuca.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-12-05)
Обработано offset 100, собрано 200 постов (последний: 2025-05-12)
Обработано offset 200, собрано 300 постов (последний: 2025-01-13)
Обработано offset 300, собрано 400 постов (последний: 2024-07-02)
Обработано offset 400, собрано 500 постов (последний: 2023-11-18)
Обработано offset 500, собрано 600 постов (последний: 2023-03-07)
Обработано offset 600, собрано 700 постов (последний: 2022-10-08)
Обработано offset 700, собрано 800 постов (последний: 2022-05-11)
Обработано offset 800, собрано 900 постов (последний: 2021-12-08)
Обработано offset 900, собрано 1000 постов (последний: 2021-05-31)
Обработано offset 1000, собрано 1100 постов (последний: 2020-11-18)

- Достигли 2019 года -
Последний пост: 2763 от 2019-12-31
Всего собрано постов: 1194
Диапазон дат: 2020-01-22 - 2026-05-04


In [ ]:
# мпсру
group_id = '-180525620'
name = 'mpsuru.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-03)
Обработано offset 100, собрано 200 постов (последний: 2025-12-05)
Обработано offset 200, собрано 300 постов (последний: 2025-10-09)
Обработано offset 300, собрано 400 постов (последний: 2025-07-11)
Обработано offset 400, собрано 500 постов (последний: 2025-05-12)
Обработано offset 500, собрано 600 постов (последний: 2025-03-10)
Обработано offset 600, собрано 700 постов (последний: 2024-12-18)
Обработано offset 700, собрано 800 постов (последний: 2024-10-16)
Обработано offset 800, собрано 900 постов (последний: 2024-08-06)
Обработано offset 900, собрано 1000 постов (последний: 2024-06-10)
Обработано offset 1000, собрано 1100 постов (последний: 2024-04-12)
Обработано offset 1100, собрано 1200 постов (последний: 2024-03-07)
Обработано offset 1200, собрано 1300 постов (последний: 2023-12-28)
Обработано offset 1300, собрано 1

In [ ]:
# шнитке
group_id = '-60900154'
name = 'shnitke.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2025-10-08)
Обработано offset 100, собрано 200 постов (последний: 2025-01-14)
Обработано offset 200, собрано 300 постов (последний: 2024-05-25)
Обработано offset 300, собрано 400 постов (последний: 2023-10-12)
Обработано offset 400, собрано 500 постов (последний: 2023-03-03)
Обработано offset 500, собрано 600 постов (последний: 2022-10-18)
Обработано offset 600, собрано 700 постов (последний: 2022-04-25)
Обработано offset 700, собрано 800 постов (последний: 2021-12-21)
Обработано offset 800, собрано 900 постов (последний: 2021-06-12)
Обработано offset 900, собрано 1000 постов (последний: 2020-10-07)
Обработано offset 1000, собрано 1100 постов (последний: 2020-04-16)

- Достигли 2019 года -
Последний пост: 1012 от 2019-11-22
Всего собрано постов: 1138
Диапазон дат: 2020-02-12 - 2026-05-03


In [ ]:
# глазунов
group_id = '-58035731'
name = 'glasunov.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-17)
Обработано offset 100, собрано 200 постов (последний: 2026-01-25)
Обработано offset 200, собрано 300 постов (последний: 2025-11-24)
Обработано offset 300, собрано 400 постов (последний: 2025-09-26)
Обработано offset 400, собрано 500 постов (последний: 2025-08-03)
Обработано offset 500, собрано 600 постов (последний: 2025-06-12)
Обработано offset 600, собрано 700 постов (последний: 2025-05-01)
Обработано offset 700, собрано 800 постов (последний: 2025-03-19)
Обработано offset 800, собрано 900 постов (последний: 2025-02-07)
Обработано offset 900, собрано 1000 постов (последний: 2024-12-25)
Обработано offset 1000, собрано 1100 постов (последний: 2024-11-12)
Обработано offset 1100, собрано 1200 постов (последний: 2024-09-27)
Обработано offset 1200, собрано 1300 постов (последний: 2024-08-27)
Обработано offset 1300, собрано 1

In [ ]:
# филарет
group_id = '-117974920'
name = 'filaret.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-01-06)
Обработано offset 100, собрано 200 постов (последний: 2025-09-29)
Обработано offset 200, собрано 300 постов (последний: 2025-04-21)
Обработано offset 300, собрано 400 постов (последний: 2024-11-25)
Обработано offset 400, собрано 500 постов (последний: 2024-06-13)
Обработано offset 500, собрано 600 постов (последний: 2023-12-29)
Обработано offset 600, собрано 700 постов (последний: 2023-06-26)
Обработано offset 700, собрано 800 постов (последний: 2022-10-06)
Обработано offset 800, собрано 900 постов (последний: 2021-12-14)
Обработано offset 900, собрано 1000 постов (последний: 2021-03-29)
Обработано offset 1000, собрано 1100 постов (последний: 2020-07-09)
Обработано offset 1100, собрано 1200 постов (последний: 2020-03-20)

- Достигли 2019 года -
Последний пост: 851 от 2019-12-31
Всего собрано постов: 1251
Диапазон дат: 20

In [ ]:
# рг иис
group_id = '-193133254'
name = 'rgiis.csv'

posts = get_posts(api_token, group_id, start_offset=0)

df = pd.DataFrame(posts)

print(f'Всего собрано постов: {len(df)}')
print(f'Диапазон дат: {df['date'].min()} - {df['date'].max()}')

df.to_csv(name, index=False, encoding='utf-8')

Начинаем сбор с offset = 0
Цель: дойти до 01.01.2020
--------------------------------------------------
Обработано offset 0, собрано 100 постов (последний: 2026-03-26)
Обработано offset 100, собрано 200 постов (последний: 2026-02-23)
Обработано offset 200, собрано 300 постов (последний: 2026-01-05)
Обработано offset 300, собрано 400 постов (последний: 2025-12-02)
Обработано offset 400, собрано 500 постов (последний: 2025-10-31)
Обработано offset 500, собрано 600 постов (последний: 2025-10-02)
Обработано offset 600, собрано 700 постов (последний: 2025-08-13)
Обработано offset 700, собрано 800 постов (последний: 2025-06-09)
Обработано offset 800, собрано 900 постов (последний: 2025-05-08)
Обработано offset 900, собрано 1000 постов (последний: 2025-04-16)
Обработано offset 1000, собрано 1100 постов (последний: 2025-03-14)
Обработано offset 1100, собрано 1200 постов (последний: 2025-01-30)
Обработано offset 1200, собрано 1300 постов (последний: 2024-12-05)
Обработано offset 1300, собрано 1

### Агрегация данных в единый датафрейм

In [ ]:
import os

# все файлы были перенесены в отдельную папку, откуда осуществляется их выгрузка
folder_path = '/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_parsed_unis_csvis'
dataframes = []

# последовательное чтение каждого файла и их присоединение к списку из Dataframe'ов
for file in os.listdir(folder_path):
  file_path = os.path.join(folder_path, file)
  df = pd.read_csv(file_path)
  df['source_nickname'] = file
  print(f'Dataframe {file} содержит {len(df)} публикаций, присоединяем их к основному Dataframe')
  dataframes.append(df)

# преобразование списка из Dataframe'ов в основной Datarame
if dataframes:
  data = pd.concat(dataframes, ignore_index=True)
  data['date'] = pd.to_datetime(data['date'])
  print('')
  print(f'Все готово, основной Dataframe содержит {len(data)} публикаций с 2020-01-01')
  print('')
  data = data.query("date < '2026-01-01'").reset_index(drop=True)
  data = data.dropna()
  print(f'Удалены публикации из 2026 и пропущенные значения, сновной Dataframe содержит {len(data)} публикаций с 2020-01-01 по 2025-12-31')

data.to_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_unis.csv', index=False)

Dataframe hse.csv содержит 7560 публикаций, присоединяем их к основному Dataframe
Dataframe mai.csv содержит 7844 публикаций, присоединяем их к основному Dataframe
Dataframe mgu.csv содержит 7728 публикаций, присоединяем их к основному Dataframe
Dataframe mfti.csv содержит 5731 публикаций, присоединяем их к основному Dataframe
Dataframe mirea.csv содержит 10851 публикаций, присоединяем их к основному Dataframe
Dataframe ranepa.csv содержит 11480 публикаций, присоединяем их к основному Dataframe
Dataframe misis.csv содержит 11334 публикаций, присоединяем их к основному Dataframe
Dataframe bmstu.csv содержит 7222 публикаций, присоединяем их к основному Dataframe
Dataframe rudn.csv содержит 6038 публикаций, присоединяем их к основному Dataframe
Dataframe sechenov.csv содержит 4777 публикаций, присоединяем их к основному Dataframe
Dataframe mgimo.csv содержит 7807 публикаций, присоединяем их к основному Dataframe
Dataframe fa.csv содержит 6542 публикаций, присоединяем их к основному Datafr

## Парсинг комментариев

### Дополнение датафрейма для выгрузки комментариев

In [40]:
# списки идентификаторов и названий csv файлов для добавления этой информации в набор данных
data = [['-25205856', 'hse.csv'],
        ['-50409684', 'mai.csv'],
        ['-78019879', 'mgu.csv'],
        ['-1177', 'reu.csv'],
        ['-932', 'mfti.csv'],
        ['-1388', 'mirea.csv'],
        ['-5398', 'ranepa.csv'],
        ['-62258607', 'misis.csv'],
        ['-40427933', 'bmstu.csv'],
        ['-1711', 'rudn.csv'],
        ['-65464437', 'sechenov.csv'],
        ['-26555975', 'mgimo.csv'],
        ['-6319', 'fa.csv'],
        ['-69589815', 'mephi.csv'],
        ['-65417', 'kutafina.csv'],
        ['-145886776', 'gitis.csv'],
        ['-71938736', 'gubkin.csv'],
        ['-53765914', 'rsmu.csv'],
        ['-51345945', 'mpei.csv'],
        ['-30321356', 'mpgu.csv'],
        ['-199859404', 'vgik.csv'],
        ['-3983', 'mgpu.csv'],
        ['-16479782', 'rggu.csv'],
        ['-33509', 'miet.csv'],
        ['-146300938', 'timacad.csv'],
        ['-23628595', 'guu.csv'],
        ['-97639426', 'mgsu.csv'],
        ['-38924', 'rguk.csv'],
        ['-23778827', 'shcukina.csv'],
        ['-31037181', 'muctr.csv'],
        ['-30431394', 'gnesin.csv'],
        ['-38029', 'synergy.csv'],
        ['-33991085', 'polytech.csv'],
        ['-38093474', 'mgppu.csv'],
        ['-1332509', 'stankin.csv'],
        ['-30016632', 'mgusit.csv'],
        ['-39683739', 'rgsu.csv'],
        ['-168484484', 'rosbiotech.csv'],
        ['-159', 'miit.csv'],
        ['-181211182', 'dipacademy.csv'],
        ['-97729', 'pstgu.csv'],
        ['-163624433', 'mgri.csv'],
        ['-211891378', 'balletacademy.csv'],
        ['-41423', 'gtsolifk.csv'],
        ['-187809308', 'rghpu.csv'],
        ['-11290126', 'inpsyho.csv'],
        ['-214354323', 'mslu.csv'],
        ['-138541024', 'madi.csv'],
        ['-103477801', 'mgavm.csv'],
        ['-36100537', 'mocsons.csv'],
        ['-46611574', 'msmsu.csv'],
        ['-471300', 'rash.csv'],
        ['-44133092', 'miigaik.csv'],
        ['-4292', 'mfua.csv'],
        ['-203483546', 'marhi.csv'],
        ['-30164925', 'pushkin.csv'],
        ['-1010597', 'mhat.csv'],
        ['-94240017', 'mgutm.csv'],
        ['-786', 'mtusi.csv'],
        ['-8990', 'guz.csv'],
        ['-32851', 'gitr.csv'],
        ['-194691927', 'vavt.csv'],
        ['-37520', 'gaugn.csv'],
        ['-160773003', 'ippolit.csv'],
        ['-152003', 'rosnou.csv'],
        ['-35003319', 'andriaka.csv'],
        ['-26099662', 'isi.csv'],
        ['-104797', 'imes.csv'],
        ['-7555704', 'shepka.csv'],
        ['-33044983', 'iile.csv'],
        ['-71768146', 'mositi.csv'],
        ['-4456', 'mosgu.csv'],
        ['-58519483', 'vitte.csv'],
        ['-29550550', 'mhpidesign.csv'],
        ['-19923410', 'uwc.csv'],
        ['-212123803', 'rgupru.csv'],
        ['-84480950', 'mmamos.csv'],
        ['-9003905', 'miuniversity.csv'],
        ['-32026522', 'mui.csv'],
        ['-82402911', 'shanin.csv'],
        ['-59762666', 'raikin.csv'],
        ['-29193864', 'mitumasi.csv'],
        ['-45732330', 'mstuca.csv'],
        ['-180525620', 'mpsuru.csv'],
        ['-60900154', 'shnitke.csv'],
        ['-58035731', 'glasunov.csv'],
        ['-117974920', 'filaret.csv'],
        ['-193133254', 'rgiis.csv'],
        ['-194894715', 'surikov.csv'],
        ['-111837730', 'rpamu.csv']]

# создание датафрейма с идентификаторами сообществ и названием файлов с их публикациями
df = pd.DataFrame(data, columns=['group_id', 'source_nickname'])
df.to_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/unis_ids.csv', index=False)

In [41]:
data = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_unis.csv')
data.head()

,post_id,date,text,likes,comments_count,reposts,views,source_nickname
0,77031,2025-12-31,🎄 Не хотим отвлекать вас от салатно-гирляндных...,42,0,5,4951,hse.csv
1,77030,2025-12-30,🎄 Чем запомнился уходящий год для Вышки?\n\nРа...,44,0,2,6790,hse.csv
2,77028,2025-12-30,🌳 Зеленая трансформация: от мифов к реалиям \n...,20,0,3,3091,hse.csv
3,77026,2025-12-30,"📖 «Тревожность», «зумер», «выгорание» — слова ...",42,0,9,14242,hse.csv
4,77022,2025-12-29,"🔍 Сценарные прогнозы, ИИ-агенты, технологии 6G...",22,0,1,3841,hse.csv


In [42]:
# добавление колонки с идентификатором сообщества
data = data.merge(df[['source_nickname', 'group_id']], on='source_nickname', how='left')

# функция для конструирования колонки, содержащей ссылку на конкретную публикацию
def construct_post_link(row):
  group_id = row['group_id'].split('-')[1]
  post_id = row['post_id']
  return f'https://vk.com/club{group_id}?w=wall-{group_id}_{post_id}'

# добавление колонки со ссылкой на публикацию сообщества
data['post_link'] = data.apply(construct_post_link, axis=1)

data.head()

,post_id,date,text,likes,comments_count,reposts,views,source_nickname,group_id,post_link
0,77031,2025-12-31,🎄 Не хотим отвлекать вас от салатно-гирляндных...,42,0,5,4951,hse.csv,-25205856,https://vk.com/club25205856?w=wall-25205856_77031
1,77030,2025-12-30,🎄 Чем запомнился уходящий год для Вышки?\n\nРа...,44,0,2,6790,hse.csv,-25205856,https://vk.com/club25205856?w=wall-25205856_77030
2,77028,2025-12-30,🌳 Зеленая трансформация: от мифов к реалиям \n...,20,0,3,3091,hse.csv,-25205856,https://vk.com/club25205856?w=wall-25205856_77028
3,77026,2025-12-30,"📖 «Тревожность», «зумер», «выгорание» — слова ...",42,0,9,14242,hse.csv,-25205856,https://vk.com/club25205856?w=wall-25205856_77026
4,77022,2025-12-29,"🔍 Сценарные прогнозы, ИИ-агенты, технологии 6G...",22,0,1,3841,hse.csv,-25205856,https://vk.com/club25205856?w=wall-25205856_77022


### Написание функции по извлечению данных комментариев с помощью VK API

In [1]:
# импорт библиотек
import vk_api
from datetime import datetime
import time
import pandas as pd

# инициализация функции для извлечения комментариев
def get_comments_from_posts(api_token, post_link):
    '''
    Получает тексты комментариев из публикаций сообществ с сайта https://vk.com.

    Параметры:
        api_token: ключ доступа для выполнения запросов к VK API
        post_link: полная ссылка на публикацию
    Вывод функции:
        Возвращает str – тексты всех комментариев к публикации, через разделитель '|'; '0' – если комментарии отсутсвуют
    '''
  # инициализация VK API
    vk_session = vk_api.VkApi(token=api_token)
    vk = vk_session.get_api()

    wall_ = post_link.split('wall')[1]
    owner_id, post_id = wall_.split('_')
    owner_id = int(owner_id)
    post_id = int(post_id)

    # инициализируем вспомогательные элементы
    all_comments = []
    offset = 0
    count = 100

    # пока все комментарии не собраны, выполняется цикл
    while True:
        # отправка запроса для получения кромментариев
        comments_response = vk.wall.getComments(owner_id=owner_id,
                                               post_id=post_id,
                                               count=count,
                                               offset=offset,
                                               need_likes=0,
                                               extended=0)
        
        if comments_response.get('items'):
            for comment in comments_response['items']:
                comment_text = comment.get('text', '')
                if comment_text and comment_text.strip():
                    all_comments.append(comment_text.strip())


            if len(comments_response['items']) < count:
                break
                
        offset += count
        # задержка алгоритма для избежания превышения установленных лимитов запросов (rps)
        time.sleep(0.05)
    # объединение комментариев в ячейку
    if all_comments:
        text = ' | '.join(all_comments)
        return text
    else:
        return '0'

/Users/ekaterinamalkova/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [9]:
from tqdm import tqdm
tqdm.pandas()

# на данном этапе приводится выполнение 2 параллели с новым api_token для ускорения работы
data_with_comments = data[data['comments_count'] > 0]

data_with_comments_3 = data_with_comments[50000:]

In [ ]:
# после пробного применения данного кода стало ясно, что необходимо производить оптимизацию, поскольку
# 1) загрузка происходила очень долго и 2) многократное появление сообщений о превышении числа rps

# comments = []
# for link in tqdm(data_with_comments['post_link']):
#     comments.append(get_comments_from_posts(api_token, link))
#     time.sleep(0.1)
# data_with_comments['comments_text'] = comments

In [ ]:
# сохранение постов с уже прогруженными комментариями для продолжения работы
processed_df = data_with_comments.iloc[:len(comments)].copy()
processed_df['comments_text'] = comments
processed_df.to_csv('processed_posts.csv', index=False)

In [ ]:
# оптимизация выгрузки заключается в следующем:
# 1) изменения для более корректного получения owner_id и post_id
# 2) использование потока для фоновой работы над данными. в случае зависания на 80 секунд, 
    # поток прекращает работу и переходит к следующей строке с данными

In [3]:
import pandas as pd
from tqdm import tqdm
import re
import time
import threading
import warnings
import vk_api


# инициализация функции для корреткного получения owner_id и post_id с обработкой возможной ошибки
def parse_wall_links(post_link):
    '''
    Извлекает идентификаторы сообщества и публикации из ссылки. 
    
    Параметры:
        post_link: полная ссылка на публикацию
    Вывод функции:
        Возвращает int – идентификаторы сообщества и публикации
    '''
    match = re.search(r'wall(-?\d+)_(\d+)', str(post_link))

    if not match:
        raise ValueError('Ссылка не распарсилась')
    owner_id = int(match.group(1))
    post_id = int(match.group(2))

    return owner_id, post_id



# инициализация функции для извлечения комментариев
def get_comments_from_posts(api_token, post_link):
    '''
    Получает тексты комментариев из публикаций сообществ с сайта https://vk.com.

    Параметры:
        api_token: ключ доступа для выполнения запросов к VK API
        post_link: полная ссылка на публикацию
    Вывод функции:
        Возвращает str – тексты всех комментариев к публикации, через разделитель '|'; '0' – если комментарии отсутсвуют
    '''
    try:
        # инициализация VK API
        vk_session = vk_api.VkApi(token=api_token)
        vk = vk_session.get_api()
    
        owner_id, post_id = parse_wall_links(post_link)
        
        # инициализируем вспомогательные элементы
        all_comments = []
        offset = 0
        count = 100
        
        # пока все комментарии не собраны, выполняется цикл
        while True:
            # отправка запроса для получения кромментариев
            comments_response = vk.wall.getComments(owner_id=owner_id,
                                                   post_id=post_id,
                                                   count=count,
                                                   offset=offset,
                                                   need_likes=0,
                                                   extended=0)
            
            if comments_response.get('items'):
                for comment in comments_response['items']:
                    comment_text = comment.get('text', '')
                    if comment_text and comment_text.strip():
                        all_comments.append(comment_text.strip())
    
    
                if len(comments_response['items']) < count:
                    break
            else:
                break
            offset += count
            # задержка алгоритма для избежания превышения установленных лимитов запросов (rps)
            time.sleep(0.45)
        # объединение комментариев в ячейку
        if all_comments:
            text = ' | '.join(all_comments)
            return text
        else:
            return '0'
    
    except Exception:
        return 'нужна проверка'

# инициализация функции защищенного вызова с таймаутом
def get_comments_thread(api_token, post_link, timeout=80):
    result = {'value': 'нужна проверка'}

    # вызывает get_comments_from_posts, если успешно, записывает результат, иначе помечает как 'нужна проверка'
    def target():
        try: 
            result['value'] = get_comments_from_posts(api_token, post_link)
        except Exception:
            result['value'] = 'нужна проверка'

    # инициализация и запуск потока
    thread = threading.Thread(target=target)
    thread.daemon = True
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        return 'нужна проверка'
    if result['value']:
        return result['value']

    else:
        return 'нужна проверка'

/Users/ekaterinamalkova/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Применение функции для выгрузки комментариев из публикаций

Данные были разделены на две части: первые 50тыс строк и оставшиеся 61тыс строк
Это было осуществлено для ускорения выгрузки данных с использованием самостоятельных процессов в отдельных ноутбуках с отдельными ключами.

Первая часть данных:

In [ ]:
warnings.filterwarnings('ignore')

# названия файлов, куда будут сохраняться промежуточные и финальные результаты
progress_file = 'progress.csv'
final_file = 'final_result.csv'

# таймаут в секундах
timeout = 80
# число постов, через которое осуществляется сохранение прогресса
save_loop = 100

# первая часть данных, начиная с 0 и до 50000 строки
data_with_comments_1 = data_with_comments.head(50000).reset_index(drop=True)

progress_df = pd.read_csv(progress_file)
done = len(progress_df)
comments = progress_df['comments'].fillna('нужна проверка').tolist()

# цикл, в котором применяются функции для извлечения комментариев с обработкой ошибок и использованием таймаута в потоке
for i, post_link in enumerate(tqdm(data_with_comments_1['post_link'].iloc[done:],
                                  desc='Cбор комментариев', initial=done,
                                  total=len(data_with_comments_1))):
    current_index = done +i

    comment = get_comments_thread(api_token=api_token, post_link=post_link, timeout=timeout)
    comments.append(comment)
    time.sleep(0.15)

    # сохранение прогресса каждые 100 постов
    if (current_index +1) % save_loop == 0:
        temp_df = data_with_comments_1.iloc[:current_index +1].copy()
        temp_df['comments'] = comments
        temp_df.to_csv(progress_file, index=False)


result_df = data_with_comments_1.iloc[:len(comments)].copy()
result_df['comments'] = comments

result_df.to_csv(progress_file, index=False)
result_df.to_csv(final_file, index=False)

print('Готово!')
print(f'Итоговый файл: {final_file}')

Вторая часть данных, которая обрабатывалась параллельно в другом ноутбуке-копии с новым ключем доступа:

In [14]:
warnings.filterwarnings('ignore')

# названия файлов, куда будут сохраняться промежуточные и финальные результаты
progress_file = 'progress_3.csv'
final_file = 'final_result_3.csv'

# таймаут в секундах
timeout = 80
# число постов, через которое осуществляется сохранение прогресса
save_loop = 100

# вторая часть данных, начиная с 50000 строки и до конца
data_with_comments_3 = data_with_comments.iloc[50000:].reset_index(drop=True)


progress_df = pd.read_csv(progress_file)
done = len(progress_df)
comments = progress_df['comments'].fillna('нужна проверка').tolist()

# цикл, в котором применяются функции для извлечения комментариев с обработкой ошибок и использованием таймаута в потоке
for i, post_link in enumerate(tqdm(data_with_comments_3['post_link'].iloc[done:],
                                  desc='Cбор комментариев', initial=done,
                                  total=len(data_with_comments_3))):
    current_index = done +i

    comment = get_comments_thread(api_token=api_token, post_link=post_link, timeout=timeout)
    comments.append(comment)
    time.sleep(0.15)

    # сохранение прогресса каждые 100 постов
    if (current_index +1) % save_loop == 0:
        temp_df = data_with_comments_3.iloc[:current_index +1].copy()
        temp_df['comments'] = comments
        temp_df.to_csv(progress_file, index=False)


result_df = data_with_comments_3.iloc[:len(comments)].copy()
result_df['comments'] = comments

result_df.to_csv(progress_file, index=False)
result_df.to_csv(final_file, index=False)

print('Готово!')
print(f'Итоговый файл: {final_file}')

Сбор комментариев: 100%|████████████████| 61387/61387 [3:45:42<00:00,  2.41it/s]


Готово!
Итоговый файл: final_result_3.csv


### Объединение комментариев и дополнительная обработка ошибок

In [149]:
# чтение и объединение двух частей данных
da1 = pd.read_csv("final_result.csv")
da2 = pd.read_csv("final_result_3.csv")

data_with_comments = pd.concat([da1, da2], ignore_index=True)

In [23]:
data_with_comments.query('comments == "нужна проверка"').count().iloc[0]

np.int64(161)


In [15]:
data_with_comments.query('comments == "нужна проверка"').source_nickname.value_counts()

source_nickname
misis.csv         45
mirea.csv         30
rosbiotech.csv    25
ranepa.csv         7
mocsons.csv        7
timacad.csv        6
guu.csv            4
isi.csv            3
shcukina.csv       3
reu.csv            3
miigaik.csv        3
surikov.csv        2
gitis.csv          2
mslu.csv           2
pushkin.csv        1
mgpu.csv           1
rgiis.csv          1
vitte.csv          1
mositi.csv         1
imes.csv           1
gitr.csv           1
mgu.csv            1
madi.csv           1
rgsu.csv           1
mfti.csv           1
stankin.csv        1
polytech.csv       1
rguk.csv           1
miet.csv           1
gubkin.csv         1
rudn.csv           1
bmstu.csv          1
mgusit.csv         1
Name: count, dtype: int64

In [18]:
warnings.filterwarnings('ignore')

# названия файлов, куда будут сохраняться промежуточные и финальные результаты
progress_file = 'data_with_comments_clean_progress.csv'
final_file = 'data_with_comments_clean.csv'

# таймаут в секундах
timeout = 20
# число постов, через которое осуществляется сохранение прогресса
save_loop = 5

# выделение индексов проблемных строк 
errors = data_with_comments['comments'].eq('нужна проверка')
error_ind = data_with_comments.index[errors].tolist()
print(f'Повторная обработка {len(error_ind)} строк')

# цикл, в котором применяются функции для извлечения комментариев с обработкой ошибок и использованием таймаута в потоке
for i, index in enumerate(tqdm(error_ind, desc='Повторная обработка ошибок')):

    post_link = data_with_comments.at[index, 'post_link']
    comment = get_comments_thread(api_token=api_token, post_link=post_link, timeout=timeout)    
    data_with_comments.at[index, 'comments'] = comment
    time.sleep(0.15)

    # сохранение прогресса каждые 5 постов
    if (i + 1) % save_loop == 0:
        data_with_comments.to_csv(progress_file, index=False)

data_with_comments.to_csv(progress_file, index=False)
data_with_comments.to_csv(final_file, index=False)

print('Готово!')
print(f'Итоговый файл: {final_file}')
print(f'Осталось строк с ошибкой: {data_with_comments["comments"].eq("нужна проверка").sum()}')

Повторная обработка 161 строк


Повторная обработка ошибок: 100%|█████████████| 161/161 [10:31<00:00,  2.62s/it] 


Готово!
Осталось строк с ошибкой: 24
Итоговый файл: data_with_comments_clean.csv


In [24]:
data_with_comments.query('comments == "нужна проверка"').post_link

25384       https://vk.com/club40427933?w=wall-40427933_327
38165     https://vk.com/club145886776?w=wall-145886776_...
38257     https://vk.com/club145886776?w=wall-145886776_...
86659      https://vk.com/club138541024?w=wall-138541024_35
89058      https://vk.com/club44133092?w=wall-44133092_1984
89059      https://vk.com/club44133092?w=wall-44133092_1975
89068      https://vk.com/club44133092?w=wall-44133092_1964
93392           https://vk.com/club32851?w=wall-32851_26058
96341     https://vk.com/club26099662?w=wall-26099662_68083
96374       https://vk.com/club26099662?w=wall-26099662_146
96392       https://vk.com/club26099662?w=wall-26099662_125
97175          https://vk.com/club104797?w=wall-104797_3889
99101        https://vk.com/club71768146?w=wall-71768146_83
103471    https://vk.com/club193133254?w=wall-193133254_300
104375            https://vk.com/club3983?w=wall-3983_12719
106677    https://vk.com/club36100537?w=wall-36100537_14...
106685     https://vk.com/club36100537?w

Посты по ссылкам выше недосягаемы: они либо удалены, либо скрыты правообладателем. Удалим их из выборки

In [27]:
data_with_comments = data_with_comments.query('comments != "нужна проверка"')

In [25]:
data_with_comments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111387 entries, 0 to 111386
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   post_id          111387 non-null  int64  
 1   date             111387 non-null  object 
 2   text             111387 non-null  object 
 3   likes            111387 non-null  int64  
 4   comments_count   111387 non-null  int64  
 5   reposts          111387 non-null  int64  
 6   views            111387 non-null  int64  
 7   source_nickname  111387 non-null  object 
 8   group_id         111387 non-null  float64
 9   post_link        111387 non-null  object 
 10  comments         111387 non-null  object 
dtypes: float64(1), int64(5), object(5)
memory usage: 9.3+ MB


Также удалим посты, комментарии к которым не содержат текста и были помечены как "0"

In [30]:
data_with_comments = data_with_comments.query('comments != "0"')

In [29]:
data_with_comments.query('comments == "0"').count()

post_id            12083
date               12083
text               12083
likes              12083
comments_count     12083
reposts            12083
views              12083
source_nickname    12083
group_id           12083
post_link          12083
comments           12083
dtype: int64

In [61]:
data_with_comments.to_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_with_comments.csv')

## Набор данных

Датафрейм, содержащий комментарии к публикациям

In [78]:
data_with_comments = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_with_comments.csv', index_col=0)
data_with_comments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99280 entries, 0 to 99279
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   post_id          99280 non-null  int64  
 1   date             99280 non-null  object 
 2   text             99280 non-null  object 
 3   likes            99280 non-null  int64  
 4   comments_count   99280 non-null  int64  
 5   reposts          99280 non-null  int64  
 6   views            99280 non-null  int64  
 7   source_nickname  99280 non-null  object 
 8   group_id         99280 non-null  float64
 9   post_link        99280 non-null  object 
 10  comments         99280 non-null  object 
dtypes: float64(1), int64(5), object(5)
memory usage: 9.1+ MB


In [80]:
data_with_comments.head()

,post_id,date,text,likes,comments_count,reposts,views,source_nickname,group_id,post_link,comments
0,77019,2025-12-29,🤨 Понимают ли нейросети каламбуры?\n\nНе очень...,16,1,4,3546,hse.csv,-25205856.0,https://vk.com/club25205856?w=wall-25205856_77019,"ДА!Вы-лучшие!Докажите, что нашего брата этой к..."
1,76983,2025-12-23,"🏅 Наука, бизнес, благотворительность, спорт: в...",57,2,1,12237,hse.csv,-25205856.0,https://vk.com/club25205856?w=wall-25205856_76983,Amazing
2,76975,2025-12-22,"🤖 ИСКРА, АЛЕКС и МИРА — лучшие ИИ-наставники\n...",31,1,2,7496,hse.csv,-25205856.0,https://vk.com/club25205856?w=wall-25205856_76975,а кто у вас младший директор?
3,76962,2025-12-19,💬 Абхазский язык теперь доступен в «Яндекс Пер...,46,1,5,6359,hse.csv,-25205856.0,https://vk.com/club25205856?w=wall-25205856_76962,🤔
4,76950,2025-12-18,✍ Вышка в MAX: новые каналы и призы для подпис...,28,2,9,9664,hse.csv,-25205856.0,https://vk.com/club25205856?w=wall-25205856_76950,"здравствуйте, а когда вы будите писать задания..."


Датафрейм, содержащий все тексты публикаций

In [81]:
data_unis = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/data_unis.csv', index_col=False)
data_unis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345006 entries, 0 to 345005
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   post_id          345006 non-null  int64 
 1   date             345006 non-null  object
 2   text             345006 non-null  object
 3   likes            345006 non-null  int64 
 4   comments_count   345006 non-null  int64 
 5   reposts          345006 non-null  int64 
 6   views            345006 non-null  int64 
 7   source_nickname  345006 non-null  object
dtypes: int64(5), object(3)
memory usage: 21.1+ MB


In [82]:
data_unis.head()

,post_id,date,text,likes,comments_count,reposts,views,source_nickname
0,77031,2025-12-31,🎄 Не хотим отвлекать вас от салатно-гирляндных...,42,0,5,4951,hse.csv
1,77030,2025-12-30,🎄 Чем запомнился уходящий год для Вышки?\n\nРа...,44,0,2,6790,hse.csv
2,77028,2025-12-30,🌳 Зеленая трансформация: от мифов к реалиям \n...,20,0,3,3091,hse.csv
3,77026,2025-12-30,"📖 «Тревожность», «зумер», «выгорание» — слова ...",42,0,9,14242,hse.csv
4,77022,2025-12-29,"🔍 Сценарные прогнозы, ИИ-агенты, технологии 6G...",22,0,1,3841,hse.csv


Вспомогательный датафрейм, содержащий названия файлов csv с выгрузками из университетов и соответствующие им идентификаторы сообществ

In [83]:
unis_ids = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/unis_ids.csv', index_col=False)
unis_ids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   group_id         90 non-null     int64 
 1   source_nickname  90 non-null     object
dtypes: int64(1), object(1)
memory usage: 1.5+ KB


In [84]:
unis_ids.head()

,group_id,source_nickname
0,-25205856,hse.csv
1,-50409684,mai.csv
2,-78019879,mgu.csv
3,-1177,reu.csv
4,-932,mfti.csv
